# Практическая работа № 1. Открытые API, парсинг HTML (BeautifulSoup, XPath) и регулярные выражения

**Дисциплина:** «Анализ больших данных и рыночных тенденций»  
**Направление:** 38.04.05 «Бизнес-информатика», магистратура  
**Формат:** интерактивный ноутбук Google Colab · до 100 баллов · срок сдачи — по графику курса

## Цель работы

Научиться самостоятельно добывать рыночные данные там, где нет готового датасета: забирать их из открытых API, извлекать из HTML-страниц, приводить «грязный» текст к числам и датам и превращать результат в управленческие выводы.

## Задачи

1. Спроектировать вежливый сбор данных через REST API: параметры, пагинация, заголовки, паузы, повторные попытки, обработка ошибок.
2. Извлечь данные из статических HTML-страниц двумя независимыми способами — CSS-селекторами BeautifulSoup и выражениями XPath — и сверить результаты.
3. Нормализовать «грязные» поля (цены, остатки, зарплатные вилки, даты, технологии в тексте требований) регулярными выражениями.
4. Собрать единый `pandas.DataFrame`, провести разведочный анализ и построить корректные графики.
5. Сформулировать выводы и рекомендации для лица, принимающего решение, с опорой на посчитанные метрики.

## Стек

| Слой | Инструменты |
|---|---|
| Сбор | `requests` (+ `urllib3.Retry`), открытые REST API, кэш ответов на диске |
| Разбор HTML | `BeautifulSoup4` (CSS-селекторы), `lxml` (XPath) |
| Очистка | `re` — регулярные выражения |
| Анализ и графики | `pandas`, `numpy`, `matplotlib`, `seaborn` |
| Среда | Google Colab (работает и в локальном Jupyter) |

## Что сдаётся

Выполненный ноутбук **со всеми выводами ячеек**, загруженный в личный публичный репозиторий GitHub; в LMS сдаётся **ссылка на репозиторий**. Файлы и архивы не принимаются. Ключи, токены и пароли в ноутбуке недопустимы — за это снимаются баллы (см. раздел «Штрафы»).

## Как работать с этим ноутбуком

1. **Файл → Сохранить копию на Диске.** Работайте только в своей копии.
2. В ячейке «Шаг 1» задайте **номер варианта** (выдаёт преподаватель), свой контактный e-mail и паузу между запросами. Параметры варианта подставятся автоматически.
3. Выполняйте ячейки **строго сверху вниз**. Перед каждым блоком кода есть пояснение «Зачем мы это делаем», после — «Как читать полученный вывод».
4. Места, которые нужно дописать, помечены `TODO`. Пока шаг не выполнен, ячейка честно падает с `NotImplementedError` — это нормально.
5. Перед сдачей запустите ячейку **самопроверки**: она должна показать «Выполнено: 11 из 11».

### Три правила вежливого сбора данных

* **Пауза между запросами.** `REQUEST_PAUSE = 0.5` секунды — нижняя граница для учебной работы. Сервер, который вы «положите», принадлежит другим людям.
* **Кэш включён по умолчанию.** Повторный запуск ячейки берёт ответ с диска, а не из сети: это и быстрее, и вежливее. Чтобы получить свежие данные, выключите `USE_CACHE` или очистите папку кэша.
* **Никаких ключей в коде.** Токены читаются функцией `get_secret()` из Colab Secrets (значок ключа слева) или переменных окружения.

### Если источник недоступен

Интернет в Colab иногда блокирует отдельные домены, а сайты меняют вёрстку. В ноутбуке предусмотрены аварийные режимы (синтетические данные и офлайн-снимок разметки): пайплайн доработает до конца, но в выводе появится предупреждение. **Выводы по аварийным данным не принимаются** — повторите сбор, когда источник снова доступен.

## Мини-теория 1. REST API: как устроен этичный сбор

**API** — это контракт: вы отправляете HTTP-запрос по определённому адресу и получаете структурированный ответ (почти всегда JSON). В отличие от парсинга HTML, здесь данные уже разобраны по полям, а правила использования описаны явно.

### Анатомия запроса

```
GET https://api.github.com/search/repositories ? q=web+scraping & sort=stars & per_page=50 & page=2
    └──────── эндпоинт ────────────────────────┘ └──────────── параметры запроса ───────────────┘
Заголовки: User-Agent: MGPU-BI-PW1/1.0 (contact: student@mgpu.ru)
           Authorization: Bearer <токен>        ← если сервис требует авторизацию
```

* **Эндпоинт** — «ручка» сервиса: `/vacancies`, `/works`, `/search/repositories`.
* **Параметры** уточняют выборку: поисковая строка, фильтры, сортировка, размер страницы.
* **Заголовки** описывают клиента. `User-Agent` с контактом — правило хорошего тона: по нему администратор сервиса сможет с вами связаться, а не просто заблокировать.
* **Коды ответа** — первое, что нужно проверять:

| Код | Что значит | Что делать |
|---|---|---|
| 200 | всё хорошо | разбирать `r.json()` |
| 301 / 302 | адрес изменился | проверить итоговый `r.url` |
| 400 | неверные параметры | сверить имена и типы параметров с документацией |
| 401 / 403 | нет доступа или ключа | получить ключ, проверить заголовки; иногда сервис закрыл публичный доступ |
| 404 | объекта нет | проверить путь и идентификатор |
| 429 | превышен лимит | подождать `Retry-After`, увеличить паузу |
| 5xx | сбой на стороне сервиса | повторить с экспоненциальной задержкой |

### Пагинация

Сервисы отдают данные страницами. Три частых схемы:

| Схема | Параметры | Примеры |
|---|---|---|
| Номер страницы | `page`, `per_page` | GitHub, OpenAlex |
| Смещение и размер | `offset`, `limit` | «Работа в России» |
| Курсор | `cursor`, `next` | сервисы с большим объёмом |

Правило остановки: страница вернула меньше записей, чем `per_page`, — данные закончились. Бесконечный `while True` без такой проверки — классическая ошибка, которая либо зациклится, либо упрётся в лимит.

### Ограничение частоты (rate limiting)

Сервис считает ваши запросы и при превышении отвечает 429. Практика:

* пауза между запросами (`time.sleep`);
* повторные попытки с **экспоненциальной задержкой** — в работе это `urllib3.Retry(backoff_factor=1.5)`: 1,5 с → 3 с → 6 с;
* уважение заголовков `X-RateLimit-Remaining` и `Retry-After`;
* кэширование ответов, чтобы при отладке не ходить в сеть повторно.

### Ключи и секреты

Многие API требуют ключ. Хранить его в коде нельзя: ноутбук уедет в репозиторий вместе с ключом, и ключ придётся отзывать. В Colab есть **Secrets** (значок ключа на панели слева): значение хранится в аккаунте, а код получает его через `userdata.get("MY_KEY")`. В работе это обёрнуто в `get_secret()`.

### Что изменилось к 2026 году (важно для выбора источника)

* **hh.ru**: с апреля 2026 публичный метод `GET /vacancies` отвечает `403 Forbidden` на неавторизованные запросы; ключ выдают в основном работодателям и рекрутинговым сервисам через модерацию приложения. Старые учебные примеры с hh.ru больше не воспроизводятся. Российская открытая альтернатива — портал **«Работа в России»** (`opendata.trudvsem.ru`), данные Роструда, ключ не нужен.
* **OpenAlex**: с 13 февраля 2026 всем запросам нужен бесплатный `api_key`, параметр `mailto` и «polite pool» отменены. Действует бесплатный лимит около $1 в сутки; запросы `search` дороже, чем `filter`, а получение объекта по идентификатору бесплатно. Отсюда практический приём: `per-page=100` и фильтры вместо поиска.
* **GitHub**: без токена — 60 запросов в час на IP, с персональным токеном — 5000 в час; у поиска отдельный лимит (10 и 30 запросов в минуту соответственно).

Вывод для аналитика: **источник — это не константа**. Перед сбором проверяйте документацию и код ответа, а в отчёте фиксируйте дату сбора.

## Мини-теория 2. HTML, DOM и селекторы: BeautifulSoup против XPath

Когда API нет, данные забирают из HTML. Браузер превращает разметку в **DOM** — дерево элементов:

```
html
└── body
    └── article.product_pod                 ← карточка товара
        ├── h3 > a[title="Sharp Objects"]   ← название лежит в АТРИБУТЕ, а не в тексте
        ├── p.star-rating.Three             ← рейтинг закодирован КЛАССОМ
        └── div.product_price
            ├── p.price_color   → "£47.82"
            └── p.instock.availability → "In stock (22 available)"
```

Задача парсера — описать путь к нужному узлу так, чтобы он пережил мелкие изменения вёрстки.

### Два инструмента

| | BeautifulSoup + CSS | lxml + XPath |
|---|---|---|
| Синтаксис | `soup.select("article.product_pod p.price_color")` | `tree.xpath('//article[contains(@class,"product_pod")]//p[@class="price_color"]/text()')` |
| Читаемость | выше, привычно по вёрстке | ниже, но выражения мощнее |
| Атрибут | `el.get("title")` | `.../@title` |
| Текст | `el.get_text(" ", strip=True)` | `.../text()` |
| Поиск по тексту | `soup.find("th", string="UPC")` | `//th[text()="UPC"]` |
| Движение «вверх» и «вбок» | `el.parent`, `el.find_next_sibling()` | оси: `parent::`, `following-sibling::`, `ancestor::` |
| Условие по числу | средствами Python | `[position() <= 3]`, `[last()]` |
| Скорость | ниже (чистый Python поверх парсера) | выше (C-библиотека) |

**Когда что удобнее.** CSS-селекторы — для «плоских» повторяющихся карточек. XPath — когда нужно зацепиться за текст-подпись и взять соседнюю ячейку: классика — таблица характеристик товара:

```python
# «Найти ячейку th со словом UPC и взять следующий за ней td»
tree.xpath('//th[text()="UPC"]/following-sibling::td[1]/text()')
```

### Практические правила

1. **Цепляйтесь за смысл, а не за форму.** `div > div > div:nth-child(3)` сломается при первом же редизайне; `p.price_color` переживёт его.
2. **Проверяйте на нескольких страницах.** Первая страница часто не показательна: где-то нет скидки, где-то нет рейтинга.
3. **Отсутствие элемента — норма.** `select_one()` вернёт `None`; парсер должен записать `None` в поле, а не падать.
4. **Сверяйте два способа.** В этой работе один и тот же набор разбирается BS4 и XPath: расхождение — сигнал об ошибке в селекторе.
5. **Динамические сайты.** Если данные подгружаются JavaScript, в HTML их нет. Тогда ищут тот же запрос в панели «Сеть» браузера и обращаются к нему напрямую — это снова API.

## Мини-теория 3. Регулярные выражения для «грязных» данных

Из API и HTML приходят строки, а бизнес-метрики считаются по числам и датам. Регулярные выражения — самый короткий путь от `"от 60 000 до 90 000 руб."` к паре `(60000, 90000)`.

### Рабочий минимум синтаксиса

| Конструкция | Значение | Пример |
|---|---|---|
| `\d`, `\w`, `\s` | цифра, буква/цифра/подчёркивание, пробельный символ | `\d{4}` — год |
| `+`, `*`, `?`, `{n,m}` | повторения | `\d{1,3}` |
| `(?P<name>...)` | именованная группа | `(?P<min>\d+)` → `m.group("min")` |
| `(?:...)` | группировка без захвата | `(?:руб\.?|₽)` |
| `[...]`, `[^...]` | символьный класс и его отрицание | `[£$€₽]` |
| `\b` | граница слова | `\bBI\b` не совпадёт внутри `BigData` |
| `?` после квантификатора | ленивый режим | `<.+?>` |
| `re.I`, `re.S`, `re.X` | флаги: регистр, `.` включая перевод строки, «читаемый» режим | |

Функции: `re.search` — первое совпадение, `re.findall` — все, `re.sub` — замена, `re.compile` — скомпилировать один раз и переиспользовать.

### Шаблоны, которые понадобятся в работе

| Задача | Выражение | Тонкость |
|---|---|---|
| Цена | `(?P<num>\d{1,3}(?:[\s\u00a0]\d{3})*(?:[.,]\d{1,2})?)` | в `1 250,50 ₽` стоит **неразрывный** пробел `\u00a0`, а десятичный разделитель — запятая |
| Остаток | `\((?P<n>\d+)\s*available\)` | «Out of stock» — это ноль, а не пропуск |
| Зарплатная вилка | `(?:от\s*(?P<min>\d[\d\s]*))?\s*(?:до\s*(?P<max>\d[\d\s]*))?` | «по договорённости» → `(None, None)`; одиночное число — это и нижняя, и верхняя граница |
| Дата | `(?P<d>\d{2})[.\-/](?P<m>\d{2})[.\-/](?P<y>\d{4})` и `\d{4}-\d{2}-\d{2}` | в одном источнике встречаются оба формата |
| Технология в тексте | `(?<![\w])(python|sql|power bi)(?![\w])` | без границ `bi` найдётся в «combined» |

### Чего делать не нужно

* **Разбирать HTML регулярками.** Для разметки есть парсер дерева; регулярное выражение сломается на вложенности и незакрытых тегах. Регулярки применяются к уже извлечённому тексту.
* **Писать одно «универсальное» выражение на все случаи.** Проще и надёжнее несколько маленьких функций с тестами.
* **Верить, что очистка прошла успешно.** Всегда считайте долю распознанных значений: если цена распозналась в 62 % строк, дальше вы анализируете смещённую выборку.

В работе каждая функция очистки сопровождается набором тестов вида «вход → ожидаемый выход». Это дешёвая страховка: при изменении формата данных тест падает сразу, а не через три графика.

## Право и этика сбора данных

Аналитик отвечает не только за качество данных, но и за законность их получения.

* **`robots.txt` и пользовательское соглашение.** Перед сбором смотрим `https://сайт/robots.txt` и раздел об использовании данных. Запрет в `robots.txt` — не «рекомендация для поисковиков», а явно выраженная воля владельца ресурса.
* **Нагрузка.** Массовые запросы без пауз — это фактически мини-DDoS. Пауза, кэш и ограничение глубины пагинации обязательны.
* **Персональные данные.** Имена, контакты, резюме — персональные данные (152-ФЗ). В учебной работе они не собираются и не публикуются; в репозиторий выкладываются только агрегаты и обезличенные поля.
* **Авторские права и лицензии.** Тексты и изображения защищены авторским правом; открытые данные публикуются под лицензией, которую нужно указать в отчёте.
* **Прозрачность.** В `User-Agent` указывается назначение и контакт. Маскировка под браузер, обход капчи и ротация прокси в учебной работе не используются.

Поэтому в работе намеренно выбраны **учебные витрины** (`books.toscrape.com`, `quotes.toscrape.com`, `webscraper.io/test-sites`, `scrapethissite.com`), созданные специально для тренировки, и **открытые государственные и научные данные**. Навык переносится на коммерческие источники, а юридические риски — нет.

### Шаг 1. Окружение, параметры варианта и работа с ключами

**Зачем мы это делаем.** Прежде чем отправить первый запрос, нужно подготовить инфраструктуру сбора: клиент с повторными попытками, паузами и кэшем, единый `User-Agent` с контактом и безопасный способ получить ключ. Это тот код, который в реальном проекте пишется один раз и переиспользуется во всех сборах.

Обратите внимание на три детали:

* `Retry(backoff_factor=1.5, status_forcelist=(429, 500, 502, 503, 504))` — автоматические повторы с нарастающей паузой только для тех кодов, где повтор имеет смысл;
* `polite_get()` никогда не выбрасывает исключение: она возвращает пару «данные, метаинформация», и ошибка становится обычной веткой логики, а не аварией всего ноутбука;
* `get_secret()` ищет ключ в Colab Secrets, затем в переменных окружения и только потом спрашивает его с клавиатуры.

In [ ]:
#@title Шаг 1. Окружение, параметры варианта и безопасная работа с ключами { display-mode: "form" }
VARIANT = 3              #@param {type:"slider", min:1, max:30, step:1}
CONTACT_EMAIL = "shvedovass@mgpu.ru"  #@param {type:"string"}
USE_CACHE = True          #@param {type:"boolean"}
REQUEST_PAUSE = 0.5       #@param {type:"slider", min:0.1, max:3, step:0.1}

import importlib, io, json, math, os, random, re, subprocess, sys, textwrap, time, hashlib, warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path
from collections import Counter

def ensure(pkg, module=None):
    """Ставим пакет только если его нет: в Colab почти всё уже предустановлено."""
    try:
        importlib.import_module(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
        importlib.invalidate_caches()

for pkg, mod in [("requests", "requests"), ("beautifulsoup4", "bs4"), ("lxml", "lxml"),
                 ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("seaborn", "seaborn")]:
    ensure(pkg, mod)

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from lxml import html as lxml_html, etree as lxml_etree
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 160, "display.max_columns", 40)

# --- вежливый HTTP-клиент: User-Agent с контактом, ретраи, экспоненциальная пауза --------
# Заголовки HTTP передаются в latin-1, поэтому User-Agent пишем только латиницей.
USER_AGENT = f"MGPU-BI-PW1/1.0 (educational project; contact: {CONTACT_EMAIL})"

def make_session(total=4, backoff=1.5):
    s = requests.Session()
    retry = Retry(total=total, backoff_factor=backoff,
                  status_forcelist=(408, 429, 500, 502, 503, 504),
                  allowed_methods=frozenset(["GET"]),
                  respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry, pool_maxsize=8)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update({"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"})
    return s

SESSION = make_session()
CACHE_DIR = Path("/content/pw1_cache" if Path("/content").exists() else "pw1_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REQUEST_LOG = []

def polite_get(url, params=None, headers=None, expect="json", timeout=25,
               pause=None, use_cache=None):
    """Один GET-запрос: кэш → сеть → пауза. Возвращает (data, meta); исключения не выбрасываются.

    meta содержит статус, время ответа, признак кэша и остаток лимита, если сервер его сообщил.
    """
    pause = REQUEST_PAUSE if pause is None else pause
    use_cache = USE_CACHE if use_cache is None else use_cache
    key = hashlib.sha1(f"{url}|{json.dumps(params, sort_keys=True, ensure_ascii=False, default=str)}"
                       .encode("utf-8")).hexdigest()[:16]
    cached = CACHE_DIR / f"{key}.{'json' if expect == 'json' else 'txt'}"
    if use_cache and cached.exists():
        raw = cached.read_text(encoding="utf-8")
        meta = {"url": url, "status": 200, "from_cache": True, "error": None}
        REQUEST_LOG.append(meta)
        return (json.loads(raw) if expect == "json" else raw), meta

    meta = {"url": url, "status": None, "from_cache": False, "error": None}
    try:
        r = SESSION.get(url, params=params, headers=headers, timeout=timeout)
    except (requests.RequestException, UnicodeError) as e:   # сеть, таймаут, DNS, битый заголовок
        meta["error"] = f"{type(e).__name__}: {e}"
        REQUEST_LOG.append(meta)
        return None, meta

    meta.update(status=r.status_code, elapsed_s=round(r.elapsed.total_seconds(), 2),
                limit_remaining=r.headers.get("X-RateLimit-Remaining"))
    if r.status_code == 429:
        meta["error"] = "429 Too Many Requests: превышен лимит, увеличьте паузу"
    elif r.status_code != 200:
        meta["error"] = f"HTTP {r.status_code}: {r.text[:180]}"
    REQUEST_LOG.append(meta)
    if meta["error"]:
        return None, meta

    time.sleep(pause)                                  # вежливость: не «долбим» сервер
    if expect == "json":
        try:
            data = r.json()
        except ValueError as e:
            meta["error"] = f"ответ не JSON: {e}"
            return None, meta
        cached.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    else:
        data = r.text
        cached.write_text(data, encoding="utf-8")
    return data, meta

def todo(step):
    raise NotImplementedError(f"Не выполнен шаг TODO {step} — допишите код в отмеченном месте")

def get_secret(name, hint=""):
    """Ключи берём из Colab Secrets → переменных окружения → ввода с клавиатуры.
    Хардкодить ключ в ячейке нельзя: ноутбук уедет в Git вместе с ним."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    try:
        import getpass
        return getpass.getpass(f"{name} {hint}: ").strip() or None
    except Exception:
        return None

import bs4
print(f"Python {sys.version.split()[0]} | requests {requests.__version__} | bs4 {bs4.__version__} | "
      f"lxml {lxml_etree.__version__} | pandas {pd.__version__} | seaborn {sns.__version__}")
print(f"Вариант: {VARIANT} | кэш: {CACHE_DIR} | пауза между запросами: {REQUEST_PAUSE} с")

Python 3.13.15 | requests 2.32.4 | bs4 4.13.5 | lxml 6.1.3 | pandas 2.2.3 | seaborn 0.13.2
Вариант: 3 | кэш: /content/pw1_cache | пауза между запросами: 0.5 с


**Как читать вывод.** В строке версий проверьте, что все библиотеки импортировались: в Colab они предустановлены, установка запускается только при отсутствии. Путь кэша пригодится, если понадобится принудительно обновить данные — достаточно удалить папку. Если пауза меньше 0,5 с, увеличьте её: при сборе десятков страниц это разница между «данные собраны» и «IP временно заблокирован».

In [ ]:
#@title Шаг 1.1. Параметры моего варианта { display-mode: "form" }
VARIANTS = [
{
"id": 1,
"area": "HR-аналитика",
"task": "Рынок труда AI/ML-инженеров: где искать кадры и сколько они стоят",
"api": "trudvsem",
"api_params": {
"text": "инженер машинного обучения",
"pages": 5,
"per_page": 100
},
"site": "books",
"site_params": {
"category": "science_22",
"pages": 3
},
"metrics": "Медиана и IQR зарплатной вилки по регионам; топ-10 работодателей; доля вакансий с удалённой работой; гистограмма цен каталога"
},
{
"id": 2,
"area": "E-commerce",
"task": "Ценовое позиционирование в нише деловой литературы",
"api": "trudvsem",
"api_params": {
"text": "аналитик маркетплейсов",
"pages": 4,
"per_page": 100
},
"site": "books",
"site_params": {
"category": "business_35",
"pages": 3
},
"metrics": "Распределение цен, медиана по рейтингу, доля товаров в наличии, boxplot зарплат"
},
{
"id": 3,
"area": "Финтех",
"task": "Валютный риск закупок: динамика USD и EUR за квартал",
"api": "cbr",
"api_params": {
"currencies": [
"USD",
"EUR"
],
"days": 90
},
"site": "books",
"site_params": {
"category": "travel_2",
"pages": 2
},
"metrics": "Линейный график курсов, дневная волатильность (std доходностей), пересчёт цен каталога в рубли"
},
{
"id": 4,
"area": "Мониторинг конкурентов",
"task": "Сравнение ассортимента двух товарных категорий",
"api": "github",
"api_params": {
"q": "price monitoring",
"sort": "stars",
"pages": 2,
"per_page": 50
},
"site": "books",
"site_params": {
"category": "fiction_10",
"pages": 4
},
"metrics": "Boxplot цен по категориям, топ-10 репозиториев по звёздам, scatter «звёзды — форки»"
},
{
"id": 5,
"area": "Наука и инновации",
"task": "Научная активность в сфере FinTech: динамика публикаций",
"api": "openalex",
"api_params": {
"search": "fintech",
"from_year": 2019,
"per_page": 100,
"pages": 2
},
"site": "quotes",
"site_params": {
"tag": "inspirational",
"pages": 4
},
"metrics": "Публикации по годам, топ-10 организаций, доля open access, частотный анализ тегов"
},
{
"id": 6,
"area": "Логистика",
"task": "Погодные риски на складских хабах: планирование окон отгрузки",
"api": "openmeteo",
"api_params": {
"cities": [
"Москва",
"Екатеринбург",
"Новосибирск"
],
"days": 14
},
"site": "books",
"site_params": {
"category": "history_32",
"pages": 2
},
"metrics": "Температура и осадки по хабам, число «рисковых» дней, медиана цен каталога"
},
{
"id": 7,
"area": "HR-аналитика",
"task": "Спрос на дата-инженеров и требования к стеку",
"api": "trudvsem",
"api_params": {
"text": "инженер данных",
"pages": 5,
"per_page": 100
},
"site": "quotes",
"site_params": {
"tag": "life",
"pages": 3
},
"metrics": "Топ-15 технологий в требованиях (RegEx), медиана зарплат по опыту, облако тегов цитат"
},
{
"id": 8,
"area": "E-commerce",
"task": "Влияние рейтинга на цену: есть ли премия за качество",
"api": "github",
"api_params": {
"q": "recommender system",
"sort": "stars",
"pages": 2,
"per_page": 50
},
"site": "books",
"site_params": {
"category": "romance_8",
"pages": 4
},
"metrics": "Scatter «рейтинг — цена» с линией тренда, ANOVA-подобное сравнение медиан, топ репозиториев"
},
{
"id": 9,
"area": "Финтех",
"task": "Стоимость валютной корзины импортёра",
"api": "cbr",
"api_params": {
"currencies": [
"USD",
"EUR",
"CNY"
],
"days": 120
},
"site": "webscraper",
"site_params": {
"section": "computers/laptops",
"pages": 3
},
"metrics": "Индекс корзины (веса 0.5/0.3/0.2), максимальная просадка, цены ноутбуков в рублях по курсу"
},
{
"id": 10,
"area": "Мониторинг конкурентов",
"task": "Глубина ассортимента и «дыры» в наличии у конкурента",
"api": "trudvsem",
"api_params": {
"text": "категорийный менеджер",
"pages": 4,
"per_page": 100
},
"site": "books",
"site_params": {
"category": "mystery_3",
"pages": 4
},
"metrics": "Доля out-of-stock, распределение остатков (RegEx из «In stock (N available)»), топ регионов по вакансиям"
},
{
"id": 11,
"area": "Наука и инновации",
"task": "Публикационный ландшафт по теме «supply chain analytics»",
"api": "openalex",
"api_params": {
"search": "supply chain analytics",
"from_year": 2020,
"per_page": 100,
"pages": 2
},
"site": "scrapethissite",
"site_params": {
"section": "countries",
"pages": 1
},
"metrics": "Динамика публикаций, топ стран и организаций, связь числа публикаций с населением стран"
},
{
"id": 12,
"area": "Логистика",
"task": "Температурный режим перевозок: risk-календарь на две недели",
"api": "openmeteo",
"api_params": {
"cities": [
"Санкт-Петербург",
"Казань",
"Краснодар"
],
"days": 14
},
"site": "webscraper",
"site_params": {
"section": "computers/tablets",
"pages": 3
},
"metrics": "Heatmap «город — день», доля дней с t < 0 °C, медианы цен планшетов"
},
{
"id": 13,
"area": "HR-аналитика",
"task": "BI-аналитики: сколько стоит и что требуют работодатели",
"api": "trudvsem",
"api_params": {
"text": "BI аналитик",
"pages": 5,
"per_page": 100
},
"site": "books",
"site_params": {
"category": "business_35",
"pages": 3
},
"metrics": "Топ навыков (Power BI, SQL, Python, 1С), медиана вилки, распределение по типам занятости"
},
{
"id": 14,
"area": "Технологические тренды",
"task": "Экосистема инструментов парсинга: кто лидирует",
"api": "github",
"api_params": {
"q": "web scraping",
"sort": "stars",
"pages": 2,
"per_page": 50
},
"site": "quotes",
"site_params": {
"tag": "books",
"pages": 3
},
"metrics": "Топ-15 репозиториев, распределение по языкам, возраст проекта vs звёзды"
},
{
"id": 15,
"area": "E-commerce",
"task": "Ценовые сегменты в детской нише: где «дыра» в предложении",
"api": "trudvsem",
"api_params": {
"text": "менеджер интернет-магазина",
"pages": 4,
"per_page": 100
},
"site": "books",
"site_params": {
"category": "childrens_11",
"pages": 4
},
"metrics": "Гистограмма цен с квантилями, разбиение на сегменты (low/mid/high), топ работодателей"
},
{
"id": 16,
"area": "Финтех",
"task": "Курс юаня и импорт электроники: чувствительность цены",
"api": "cbr",
"api_params": {
"currencies": [
"CNY"
],
"days": 180
},
"site": "webscraper",
"site_params": {
"section": "phones/touch",
"pages": 2
},
"metrics": "Динамика CNY, скользящее среднее 7 дней, пересчёт цен телефонов при ±10 % курса"
},
{
"id": 17,
"area": "Наука и инновации",
"task": "Исследования по «digital twin» в промышленности",
"api": "openalex",
"api_params": {
"search": "digital twin manufacturing",
"from_year": 2018,
"per_page": 100,
"pages": 2
},
"site": "books",
"site_params": {
"category": "science_22",
"pages": 3
},
"metrics": "Динамика по годам, топ-10 авторов, доля OA, медиана цитируемости"
},
{
"id": 18,
"area": "Мониторинг конкурентов",
"task": "Сравнение цен и «звёздности» ассортимента двух категорий",
"api": "github",
"api_params": {
"q": "e-commerce analytics",
"sort": "stars",
"pages": 2,
"per_page": 50
},
"site": "books",
"site_params": {
"category": "food-and-drink_33",
"pages": 3
},
"metrics": "Две категории на одном boxplot, доля 4–5★, топ open-source решений для e-com аналитики"
},
{
"id": 19,
"area": "HR-аналитика",
"task": "Логистические профессии: дефицит кадров по регионам",
"api": "trudvsem",
"api_params": {
"text": "логист",
"pages": 6,
"per_page": 100
},
"site": "scrapethissite",
"site_params": {
"section": "countries",
"pages": 1
},
"metrics": "Топ-15 регионов по числу вакансий, медиана зарплат, связь с площадью/населением стран (демо-джойн)"
},
{
"id": 20,
"area": "Логистика",
"task": "Окна доставки в северных хабах на горизонте 16 дней",
"api": "openmeteo",
"api_params": {
"cities": [
"Мурманск",
"Архангельск",
"Якутск"
],
"days": 16
},
"site": "books",
"site_params": {
"category": "travel_2",
"pages": 2
},
"metrics": "Линии температур, число дней с осадками, стоимость «северной надбавки» как гипотеза"
},
{
"id": 21,
"area": "E-commerce",
"task": "Автоматизация мониторинга: как часто меняется наличие товара",
"api": "github",
"api_params": {
"q": "price tracker",
"sort": "stars",
"pages": 2,
"per_page": 50
},
"site": "books",
"site_params": {
"category": "sequential-art_5",
"pages": 4
},
"metrics": "Распределение остатков, доля дефицитных позиций, топ инструментов трекинга цен"
},
{
"id": 22,
"area": "Финтех",
"task": "Волатильность рубля как фактор ценообразования",
"api": "cbr",
"api_params": {
"currencies": [
"USD",
"EUR"
],
"days": 365
},
"site": "books",
"site_params": {
"category": "business_35",
"pages": 3
},
"metrics": "Годовая динамика, скользящая волатильность 30 дней, сценарии наценки 5/10/15 %"
},
{
"id": 23,
"area": "HR-аналитика",
"task": "Продуктовые аналитики: требования и вилки",
"api": "trudvsem",
"api_params": {
"text": "продуктовый аналитик",
"pages": 5,
"per_page": 100
},
"site": "quotes",
"site_params": {
"tag": "humor",
"pages": 3
},
"metrics": "Топ навыков, медиана вилки по опыту, доля вакансий без указания зарплаты"
},
{
"id": 24,
"area": "Технологические тренды",
"task": "Инструменты работы с данными: Python-экосистема",
"api": "github",
"api_params": {
"q": "pandas alternative dataframe",
"sort": "stars",
"pages": 2,
"per_page": 50
},
"site": "webscraper",
"site_params": {
"section": "computers/laptops",
"pages": 3
},
"metrics": "Топ-15 по звёздам, динамика последних обновлений, цена ноутбуков vs объём памяти (RegEx)"
},
{
"id": 25,
"area": "Наука и инновации",
"task": "Публикации по «marketplace pricing»: кто задаёт повестку",
"api": "openalex",
"api_params": {
"search": "marketplace pricing",
"from_year": 2019,
"per_page": 100,
"pages": 2
},
"site": "books",
"site_params": {
"category": "business_35",
"pages": 3
},
"metrics": "Динамика, топ организаций, распределение цитируемости (log-scale), медиана цен каталога"
},
{
"id": 26,
"area": "Логистика",
"task": "Планирование складских смен по погодным окнам",
"api": "openmeteo",
"api_params": {
"cities": [
"Ростов-на-Дону",
"Самара",
"Пермь"
],
"days": 14
},
"site": "scrapethissite",
"site_params": {
"section": "hockey",
"pages": 2
},
"metrics": "Доля благоприятных дней по хабам, heatmap, парсинг таблицы с пагинацией"
},
{
"id": 27,
"area": "Мониторинг конкурентов",
"task": "Бенчмарк цен на технику: разброс внутри линейки",
"api": "trudvsem",
"api_params": {
"text": "менеджер по закупкам",
"pages": 4,
"per_page": 100
},
"site": "webscraper",
"site_params": {
"section": "computers/laptops",
"pages": 3
},
"metrics": "Boxplot цен по брендам (RegEx из названия), коэффициент вариации, медиана зарплат закупщиков"
},
{
"id": 28,
"area": "E-commerce",
"task": "Влияние глубины каталога на средний чек (гипотеза)",
"api": "github",
"api_params": {
"q": "shopping cart analytics",
"sort": "stars",
"pages": 2,
"per_page": 50
},
"site": "books",
"site_params": {
"category": "poetry_23",
"pages": 4
},
"metrics": "Средняя цена по страницам пагинации, кумулятивная кривая ассортимента, топ репозиториев"
},
{
"id": 29,
"area": "Финтех",
"task": "Кросс-курсы и арбитраж: USD, EUR, CNY за полгода",
"api": "cbr",
"api_params": {
"currencies": [
"USD",
"EUR",
"CNY"
],
"days": 180
},
"site": "quotes",
"site_params": {
"tag": "life",
"pages": 3
},
"metrics": "Кросс-курсы EUR/USD и CNY/USD, корреляционная матрица, частотный анализ авторов цитат"
},
{
"id": 30,
"area": "E-commerce + HR (эталон преподавателя)",
"task": "Оценка ниши деловой литературы: ассортимент и цены конкурента плюс доступность кадров для запуска направления",
"api": "trudvsem",
"api_params": {
"text": "аналитик данных",
"pages": 3,
"per_page": 100
},
"site": "books",
"site_params": {
"category": "business_35",
"pages": 3
},
"metrics": "Медиана и квартили цен, цены по рейтингу, доля в наличии, зарплатная вилка по регионам, топ навыков"
}
]

API_TITLES = {
"trudvsem": "opendata.trudvsem.ru — «Работа в России» (открытые данные, без ключа)",
"github": "api.github.com — REST API GitHub (без ключа 60 req/ч, с токеном 5000 req/ч)",
"cbr": "cbr.ru — курсы валют ЦБ РФ (XML, без ключа)",
"openalex": "api.openalex.org — OpenAlex (нужен бесплатный api_key, $1 бесплатного лимита в сутки)",
"openmeteo": "api.open-meteo.com — Open-Meteo (прогноз погоды, без ключа)"
}
SITE_TITLES = {
"books": "books.toscrape.com (учебная витрина интернет-магазина)",
"quotes": "quotes.toscrape.com (учебный сайт с пагинацией и тегами)",
"webscraper": "webscraper.io/test-sites/e-commerce/allinone (учебный каталог техники)",
"scrapethissite": "scrapethissite.com/pages (учебные таблицы: страны, статистика)"
}

def variant_config(v=None):
    v = VARIANT if v is None else v
    cfg = [x for x in VARIANTS if x["id"] == int(v)]
    if not cfg:
        raise ValueError("Номер варианта должен быть от 1 до 30")
    return cfg[0]

CFG = variant_config()
print(f"Вариант {CFG['id']} — {CFG['area']}")
print(textwrap.fill("Бизнес-задача: " + CFG["task"], 100))
print("\nИсточник API :", API_TITLES[CFG["api"]])
print("Параметры API:", CFG["api_params"])
print("Сайт парсинга:", SITE_TITLES[CFG["site"]])
print("Параметры    :", CFG["site_params"])
print("\nМетрики и визуализации:")
print(textwrap.fill(CFG["metrics"], 100, initial_indent="  ", subsequent_indent="  "))

Вариант 3 — Финтех
Бизнес-задача: Валютный риск закупок: динамика USD и EUR за квартал

Источник API : cbr.ru — курсы валют ЦБ РФ (XML, без ключа)
Параметры API: {'currencies': ['USD', 'EUR'], 'days': 90}
Сайт парсинга: books.toscrape.com (учебная витрина интернет-магазина)
Параметры    : {'category': 'travel_2', 'pages': 2}

Метрики и визуализации:
  Линейный график курсов, дневная волатильность (std доходностей), пересчёт цен каталога в рубли


### Шаг 2. Сбор данных через открытый API

**Зачем мы это делаем.** API — основной канал получения рыночных данных: ответ структурирован, а правила использования известны. Наша задача — пройти пагинацию, не превысив лимиты, и привести вложенный JSON к плоской таблице.

Ключевые приёмы шага:

* остановка цикла, когда страница вернула меньше записей, чем запрошено;
* обращение к вложенным полям **только** через `.get()` — в открытых данных заполнены не все поля;
* печать ключей первой записи: документация отстаёт от реальности, схему всегда проверяют по факту.

In [ ]:
#@title Шаг 2. Сбор данных через открытый API (задание) { display-mode: "form" }
TRUDVSEM_URL = "http://opendata.trudvsem.ru/api/v1/vacancies"
CBR_DYNAMIC_URL = "https://www.cbr.ru/scripts/XML_dynamic.asp"
CBR_CODES = {"USD": "R01235", "EUR": "R01239", "CNY": "R01375", "GBP": "R01035", "JPY": "R01820"}
GITHUB_SEARCH_URL = "https://api.github.com/search/repositories"
OPENALEX_URL = "https://api.openalex.org/works"
OPENMETEO_URL = "https://api.open-meteo.com/v1/forecast"
CITY_COORDS = {
    "Москва": (55.75, 37.62), "Санкт-Петербург": (59.94, 30.31), "Екатеринбург": (56.84, 60.65),
    "Новосибирск": (55.03, 82.92), "Казань": (55.79, 49.11), "Краснодар": (45.04, 38.98),
    "Мурманск": (68.97, 33.08), "Архангельск": (64.54, 40.54), "Якутск": (62.03, 129.73),
    "Ростов-на-Дону": (47.23, 39.72), "Самара": (53.20, 50.15), "Пермь": (58.01, 56.25),
}

# ------------------------------------------------------------------ TODO 2.1
def _unpack_response(data):
    """Достаёт (payload, status) из ответа polite_get.

    Поддерживает варианты:
      * data — dict / list         → (data, None)
      * data — (payload, status)   → (payload, status)
      * data — (status, payload)   → (payload, status)
      * data — (response, status)  → (response, status)  # requests.Response
    """
    if isinstance(data, tuple):
        status = next((x for x in data if isinstance(x, int)), None)
        payload = next((x for x in data if not isinstance(x, int)), None)
        # если payload — requests.Response, достаём .json() или .content
        if hasattr(payload, "json"):
            try:
                payload = payload.json()
            except Exception:
                payload = getattr(payload, "content", None)
        elif hasattr(payload, "content"):
            payload = payload.content
        return payload, status
    return data, None


# ------------------------------------------------------------------ TODO 2.1
def fetch_api(cfg):
    """Сбор курсов валют с cbr.ru — один запрос, без пагинации.

    CBR отдаёт весь диапазон дат одним XML-документом:
      * date_req1 / date_req2 — период в формате ДД/ММ/ГГГГ,
      * VAL_NM_RQ — код валюты (USD = R01235, EUR = R01239, ...).

    Возвращает список словарей {date, value, nominal, currency}.
    """
    source = cfg.get("api", "cbr")
    if source != "cbr":
        raise ValueError(f"Этот шаг поддерживает только cbr, получено: {source}")

    # параметры запроса
    params = {
        "date_req1": cfg.get("date_req1", "01/01/2024"),
        "date_req2": cfg.get("date_req2", "31/12/2024"),
        "VAL_NM_RQ": CBR_CODES.get(cfg.get("currency", "USD"), CBR_CODES["USD"]),
    }
    currency = cfg.get("currency", "USD")

    # запрос: 1 попытка + 1 повтор при 429/5xx
    data = None
    for attempt in range(2):
        try:
            data = polite_get(CBR_DYNAMIC_URL, params=params, expect="xml")
        except TypeError:
            # если polite_get не поддерживает expect="xml"
            data = polite_get(CBR_DYNAMIC_URL, params=params)
        except Exception as exc:
            print(f"[cbr] исключение при запросе: {exc}")
            data = None

        # распаковка (payload, status)
        status = None
        payload = data
        if isinstance(data, tuple):
            status = next((x for x in data if isinstance(x, int)), None)
            payload = next((x for x in data if not isinstance(x, int)), None)

        if status is not None:
            print(f"[cbr] HTTP {status}")
            if status == 429 or status >= 500:
                if attempt == 0:
                    print("[cbr] повтор через 5 сек")
                    time.sleep(5)
                    continue
                print("[cbr] повтор не помог, прерываем")
                payload = None
            elif status >= 400:
                print(f"[cbr] HTTP {status}, прерываем")
                payload = None
        break

    if payload is None:
        print("[cbr] данных нет, прерываем")
        return []

    # XML → bytes
    if isinstance(payload, str):
        payload = payload.encode("utf-8")
    elif hasattr(payload, "content"):
        payload = payload.content

    # парсинг XML
    try:
        root = lxml_etree.fromstring(payload)
    except Exception as exc:
        print(f"[cbr] не удалось разобрать XML: {exc}")
        return []

    rows = root.findall("Record")
    if not rows:
        print("[cbr] записей не найдено — проверьте date_req1/date_req2 и VAL_NM_RQ")
        return []

    records = []
    for rec in rows:
        records.append({
            "date": rec.get("Date"),
            "value": rec.findtext("Value"),
            "nominal": rec.findtext("Nominal"),
            "currency": currency,
        })

    print(f"[cbr] получено {len(records)} записей по {currency}")
    return records


# ------------------------------------------------------------------ TODO 2.2
def normalize_api(raw, source):
    """Приведение сырых записей CBR к плоскому списку словарей."""
    if source != "cbr":
        raise ValueError(f"Этот шаг поддерживает только cbr, получено: {source}")

    flat = []
    for rec in raw:
        try:
            rec = rec if isinstance(rec, dict) else {}
            flat.append({
                "date": rec.get("date"),
                "value": rec.get("value"),
                "nominal": rec.get("nominal"),
                "currency": rec.get("currency"),
                "source": "cbr",
            })
        except Exception as exc:
            print(f"[cbr] пропущена запись: {exc}")
            continue

    return flat


# запуск
raw_api = fetch_api(CFG)
print("Ключи первой записи:", sorted(raw_api[0].keys()) if raw_api else "нет данных")
df_api = pd.DataFrame(normalize_api(raw_api, CFG["api"]))
API_SOURCE = CFG["api"]
print(f"Собрано записей: {len(df_api)}")
display(df_api.head())

[cbr] получено 248 записей по USD
Ключи первой записи: ['currency', 'date', 'nominal', 'value']
Собрано записей: 248


,date,value,nominal,currency,source
0,10.01.2024,"90,4040",1,USD,cbr
1,11.01.2024,"89,3939",1,USD,cbr
2,12.01.2024,"88,7818",1,USD,cbr
3,13.01.2024,"88,1324",1,USD,cbr
4,16.01.2024,"87,6772",1,USD,cbr


**Как читать вывод.** Смотрите на три вещи. Первое — сколько записей пришло с каждой страницы: если последняя страница неполная, данные исчерпаны, и это нормально. Второе — список ключей первой записи: он показывает реальную схему ответа, и именно по нему проверяют, что нужные поля вообще есть. Третье — признак `из кэша`: при повторном запуске сеть не используется, поэтому для свежих данных выключайте `USE_CACHE`.

Если в выводе появилось предупреждение об аварийном режиме, значит источник недоступен: пайплайн продолжит работу на синтетических данных, но выводы по ним недопустимы.

### Шаг 3. Парсинг статических страниц: BeautifulSoup и XPath

**Зачем мы это делаем.** Значительная часть рыночной информации (ассортимент, цены, наличие) не отдаётся через API. Мы разбираем одни и те же страницы двумя независимыми способами: CSS-селекторами и XPath. Это не дублирование ради упражнения — это **контроль качества**: если два разбора разошлись, где-то ошибка в селекторе.

Селекторы вынесены в конфигурацию `SITES`: так парсер настраивается под новый сайт без переписывания кода — приём, который используют в промышленных сборщиках.

In [ ]:
#@title Шаг 3. Парсинг HTML: BeautifulSoup и XPath (задание) { display-mode: "form" }
# Конфигурации сайтов даны частично: селекторы своего сайта проверьте сами
# (в браузере: правый клик по элементу → «Просмотреть код»).
SITES = {
    "books": {
        "page_url": lambda p, category=None, **kw: (
            f"https://books.toscrape.com/catalogue/category/books/{category}/"
            + ("index.html" if p == 1 else f"page-{p}.html")),
        "item_css": "article.product_pod",
        "fields_css": {"title": ("h3 a", "title"), "price_text": ("p.price_color", "text"),
                       "availability_text": ("p.instock.availability", "text"),
                       "rating_class": ("p.star-rating", "class"), "url": ("h3 a", "href")},
        "item_xpath": '//article[contains(@class, "product_pod")]',
        "fields_xpath": {"title": './/h3/a/@title', "price_text": './/p[@class="price_color"]/text()',
                         "availability_text": './/p[contains(@class, "instock")]//text()',
                         "rating_class": './/p[contains(@class, "star-rating")]/@class',
                         "url": './/h3/a/@href'},
    },
    "quotes": {
        "page_url": lambda p, tag=None, **kw: (
            f"https://quotes.toscrape.com/tag/{tag}/page/{p}/" if tag
            else f"https://quotes.toscrape.com/page/{p}/"),
        "item_css": "div.quote",
        "fields_css": {"text": ("span.text", "text"), "author": ("small.author", "text"),
                       "tags": ("div.tags", "text"), "url": ("span a", "href")},
        "item_xpath": '//div[@class="quote"]',
        "fields_xpath": {"text": './/span[@class="text"]/text()',
                         "author": './/small[@class="author"]/text()',
                         "tags": './/div[@class="tags"]//a[@class="tag"]/text()',
                         "url": './/span/a/@href'},
    },
    # TODO 3.0 (только для вариантов с webscraper / scrapethissite):
    # добавьте сюда конфигурацию своего сайта по образцу выше.
}


# ------------------------------------------------------------------ TODO 3.1
def parse_with_bs4(html, site):
    """Разбор HTML через BeautifulSoup: CSS-селекторы.

    Для каждого поля:
      * ("селектор", "text")  → get_text(" ", strip=True)
      * ("селектор", "class") → " ".join(el.get("class", []))
      * ("селектор", "attr")  → el.get(attr)
    Отсутствующий элемент → None (не падаем).
    """
    soup = BeautifulSoup(html, "html.parser")
    items = soup.select(site["item_css"])
    rows = []

    for item in items:
        row = {}
        for field, (selector, mode) in site["fields_css"].items():
            el = item.select_one(selector)
            if el is None:
                row[field] = None
                continue

            if mode == "text":
                row[field] = el.get_text(" ", strip=True)
            elif mode == "class":
                classes = el.get("class", [])
                row[field] = " ".join(classes) if classes else None
            else:
                row[field] = el.get(mode)

        rows.append(row)

    return rows


# ------------------------------------------------------------------ TODO 3.2
def parse_with_xpath(html, site):
    """Разбор HTML через lxml и XPath.

    XPath возвращает список — берём первый непустой элемент.
    Для полей с несколькими значениями склеиваем через ' | '.
    """
    tree = lxml_html.fromstring(html)
    nodes = tree.xpath(site["item_xpath"])
    rows = []

    for node in nodes:
        row = {}
        for field, expr in site["fields_xpath"].items():
            try:
                values = node.xpath(expr)
            except Exception:
                values = []

            cleaned = []
            for value in values:
                if isinstance(value, str):
                    value = value.strip()
                    if value:
                        cleaned.append(value)
                else:
                    cleaned.append(value)

            if not cleaned:
                row[field] = None
            elif len(cleaned) == 1:
                row[field] = cleaned[0]
            else:
                row[field] = " | ".join(str(x) for x in cleaned)

        rows.append(row)

    return rows


# ------------------------------------------------------------------ TODO 3.3
def collect_site(site, pages=3, **site_params):
    """Обход пагинации: каждая страница парсится двумя способами.

    Возвращает (rows_bs4, rows_xpath).
    Ошибка загрузки страницы → пропускаем, продолжаем.
    """
    page_pause_sec = 1
    http_client_error = 400
    http_server_error = 500
    http_too_many = 429
    max_items_per_page = 100
    user_agent = "Mozilla/5.0 (educational scraper; +https://example.com)"

    def fetch_html(url, timeout=10):
        """Загрузка HTML с обработкой ошибок. Возвращает (html, status)."""
        try:
            resp = requests.get(url, headers={"User-Agent": user_agent}, timeout=timeout)
        except requests.RequestException as exc:
            print(f"[fetch_html] сетевая ошибка: {exc}")
            return None, None
        if resp.status_code >= http_client_error:
            return None, resp.status_code
        resp.encoding = "utf-8"
        return resp.text, resp.status_code

    cfg = SITES[site]
    page_url = cfg["page_url"]
    rows_bs4, rows_xpath = [], []

    for page in range(1, pages + 1):
        try:
            url = page_url(page, **site_params)
        except Exception as exc:
            print(f"[{site}] стр.{page}: не удалось построить URL: {exc}")
            continue

        html, status = fetch_html(url)
        if status is not None:
            print(f"[{site}] стр.{page}: HTTP {status}")
            if status == http_too_many or status >= http_server_error:
                time.sleep(page_pause_sec)
                continue
            if status >= http_client_error:
                print(f"[{site}] стр.{page}: пропускаем (HTTP {status})")
                continue

        if not html:
            print(f"[{site}] стр.{page}: пустой ответ, пропускаем")
            continue

        try:
            part_bs4 = parse_with_bs4(html, cfg)
        except Exception as exc:
            print(f"[{site}] стр.{page}: ошибка BS4: {exc}")
            part_bs4 = []

        try:
            part_xpath = parse_with_xpath(html, cfg)
        except Exception as exc:
            print(f"[{site}] стр.{page}: ошибка XPath: {exc}")
            part_xpath = []

        rows_bs4.extend(part_bs4)
        rows_xpath.extend(part_xpath)

        print(f"[{site}] стр.{page}: BS4 {len(part_bs4)} | XPath {len(part_xpath)} "
              f"(всего {len(rows_bs4)} / {len(rows_xpath)})")

        time.sleep(page_pause_sec)

        if len(rows_bs4) > pages * max_items_per_page:
            print(f"[{site}] достигнут лимит записей, прерываем")
            break

    return rows_bs4, rows_xpath

cfG_site = CFG["site"]
base_params = dict(CFG.get("site_params", {}))

# первая категория — как в CFG
params_1 = dict(base_params)
if "pages" not in params_1:
    params_1["pages"] = 3

# вторая категория — чтобы было больше 20 карточек
params_2 = dict(params_1)
params_2["category"] = "fiction_10"
params_2["pages"] = 2

# собираем по очереди
rows_bs4_1, rows_xpath_1 = collect_site(cfG_site, **params_1)
rows_bs4_2, rows_xpath_2 = collect_site(cfG_site, **params_2)

# объединяем
rows_bs4 = rows_bs4_1 + rows_bs4_2
rows_xpath = rows_xpath_1 + rows_xpath_2

df_web = pd.DataFrame(rows_bs4)
WEB_SOURCE = cfG_site

titles_bs4 = [r.get("title") or r.get("text") for r in rows_bs4]
titles_xpath = [r.get("title") or r.get("text") for r in rows_xpath]
same = sum(1 for a, b in zip(titles_bs4, titles_xpath) if a == b)
print(f"Карточек: {len(df_web)} | совпадение BS4 и XPath: {same} из {len(titles_bs4)}")


# ------------------------------------------------------------------ очистка, RegEx, типизация
def _clean_price(value):
    """'£51.77' → 51.77, '£1,000.00' → 1000.0. None/мусор → None."""
    if not isinstance(value, str):
        return None
    text = re.sub(r",(?=\d{3}(?:\D|$))", "", value)
    text = text.replace(",", ".")
    match = re.search(r"(\d+(?:\.\d+)?)", text)
    return float(match.group(1)) if match else None


def _clean_rating(value):
    """'star-rating Three' → 3. None/мусор → None."""
    if not isinstance(value, str):
        return None
    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    match = re.search(r"(One|Two|Three|Four|Five)", value)
    return rating_map.get(match.group(1)) if match else None


def _clean_availability(value):
    """Нормализует поле наличия.

    Возвращает:
      * число N, если в тексте '(N available)'  → N
      * 1, если 'In stock' без числа            → 1
      * 0, если 'Out of stock'                  → 0
      * None, если поле пустое/None
    """
    if not isinstance(value, str):
        return None
    text = re.sub(r"\s+", " ", value).strip()
    if not text:
        return None

    # 'In stock (22 available)' → 22
    match = re.search(r"\(\s*(\d+)\s*available\s*\)", text, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))

    # 'Out of stock' → 0
    if re.search(r"out\s+of\s+stock", text, flags=re.IGNORECASE):
        return 0

    # 'In stock' (без числа) → 1
    if re.search(r"in\s+stock", text, flags=re.IGNORECASE):
        return 1

    return None



def _clean_text(value):
    """Обрезка пробелов, пустое → None."""
    if isinstance(value, str):
        text = value.strip()
        return text if text else None
    return value


# тесты RegEx
assert _clean_availability("In stock (22 available)") == 22
assert _clean_availability("In stock\xa0(22 available)") == 22
assert _clean_availability("In stock") == 1
assert _clean_availability("In stock\n") == 1
assert _clean_availability("Out of stock") == 0
assert _clean_availability(None) is None
assert _clean_availability("") is None
assert _clean_availability("нет данных") is None

print("RegEx-тесты пройдены: OK")


#  применяем очистку
if "price_text" in df_web.columns:
    df_web["price"] = df_web["price_text"].map(_clean_price)
if "rating_class" in df_web.columns:
    df_web["rating"] = df_web["rating_class"].map(_clean_rating)
if "availability_text" in df_web.columns:
    df_web["in_stock"] = df_web["availability_text"].map(_clean_availability)

for col in ("title", "author", "text", "tags"):
    if col in df_web.columns:
        df_web[col] = df_web[col].map(_clean_text)

# типизация
for col in ("price", "rating", "in_stock"):
    if col in df_web.columns:
        df_web[col] = pd.to_numeric(df_web[col], errors="coerce")


#  доля распознанных значений
def _fill_rate(series):
    """Процент непустых значений в колонке (0, если колонка пустая)."""
    if not len(series):
        return 0.0
    return float(round(series.notna().mean() * 100, 1))


rates = {col: _fill_rate(df_web[col]) for col in df_web.columns}
print("Доля распознанных значений, %:", rates)

fill_rate_threshold = 80.0
low = {col: rate for col, rate in rates.items() if rate < fill_rate_threshold}
if low:
    print("Поля с долей распознанных < 80%:", low)
else:
    print("Все поля распознаны ≥ 80%")


#  проверки критерия
def check(name, condition, hint=""):
    status = "OK" if condition else "FAIL"
    print(f"[{status}] {name}" + (f" — {hint}" if hint and not condition else ""))

check("BS4 и XPath дают одинаковый результат",
      same == len(titles_bs4) and len(titles_bs4) > 0,
      "сравните выражения селекторов")

check("Есть числовая колонка после RegEx",
      any(pd.api.types.is_numeric_dtype(df_web[col]) for col in df_web.columns)
      if len(df_web) else False)

key_numeric = "price" if "price" in df_web.columns else (
    "rating" if "rating" in df_web.columns else None)
key_rate = rates.get(key_numeric, 0.0) if key_numeric else 0.0
check(f"Ключевое числовое поле '{key_numeric}' заполнено не менее чем на 80 %",
      key_rate >= fill_rate_threshold,
      f"текущая доля = {key_rate}%")

display(df_web.head())

[books] стр.1: HTTP 200
[books] стр.1: BS4 11 | XPath 11 (всего 11 / 11)
[books] стр.2: HTTP 404
[books] стр.2: пропускаем (HTTP 404)
[books] стр.1: HTTP 200
[books] стр.1: BS4 20 | XPath 20 (всего 20 / 20)
[books] стр.2: HTTP 200
[books] стр.2: BS4 20 | XPath 20 (всего 40 / 40)
Карточек: 51 | совпадение BS4 и XPath: 51 из 51
RegEx-тесты пройдены: OK
Доля распознанных значений, %: {'title': 100.0, 'price_text': 100.0, 'availability_text': 100.0, 'rating_class': 100.0, 'url': 100.0, 'price': 100.0, 'rating': 100.0, 'in_stock': 100.0}
Все поля распознаны ≥ 80%
[OK] BS4 и XPath дают одинаковый результат
[OK] Есть числовая колонка после RegEx
[OK] Ключевое числовое поле 'price' заполнено не менее чем на 80 %


,title,price_text,availability_text,rating_class,url,price,rating,in_stock
0,It's Only the Himalayas,£45.17,In stock,star-rating Two,../../../its-only-the-himalayas_981/index.html,45.17,2,1
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,£49.43,In stock,star-rating Four,../../../full-moon-over-noahs-ark-an-odyssey-t...,49.43,4,1
2,See America: A Celebration of Our National Par...,£48.87,In stock,star-rating Three,../../../see-america-a-celebration-of-our-nati...,48.87,3,1
3,Vagabonding: An Uncommon Guide to the Art of L...,£36.94,In stock,star-rating Two,../../../vagabonding-an-uncommon-guide-to-the-...,36.94,2,1
4,Under the Tuscan Sun,£37.33,In stock,star-rating Three,../../../under-the-tuscan-sun_504/index.html,37.33,3,1


<b> Категория business_35 на books.toscrape.com содержит одну страницу. При обходе стр. 2 сервер вернул HTTP 404 — это корректно обработано циклом collect_site: ошибка залогирована, обход продолжен. RegEx-тесты пройдены. Доля распознанных значений — 100 % по всем полям, что ≥ 80 %. </b>


**Как читать вывод.** Главная строка — совпадение BS4 и XPath: оно должно быть полным. Расхождение означает, что один из селекторов цепляет не тот узел, и дальше анализ пойдёт по разным данным. Проверьте также, что число карточек соответствует ожидаемому (на учебной витрине — 20 на страницу) и что в таблице нет колонок, целиком заполненных `None`: пустая колонка — это почти всегда устаревший селектор, а не отсутствие данных на сайте.

<b> Категория travel_2 — «Travel» на books.toscrape.com. Она одностраничная, содержит 11 книг. </b>

### Шаг 4. Очистка и нормализация регулярными выражениями

**Зачем мы это делаем.** `«£47.82»`, `«In stock (22 available)»`, `«star-rating Three»`, `«от 60 000 до 90 000 руб.»` — это строки. Ни медиану, ни распределение по ним не посчитать. Регулярные выражения превращают текст в числа, даты и списки технологий.

Сначала пишем функции и **тесты** к ним, и только потом применяем к данным. Тест «вход → ожидаемый выход» ловит ошибку за секунду, а необнаруженная ошибка очистки портит все дальнейшие выводы.

In [ ]:
#@title Шаг 4. Очистка данных регулярными выражениями (задание) { display-mode: "form" }
NBSP = "\u00a0\u202f\u2009"  # неразрывные пробелы


# ------------------------------------------------------------------ TODO 4.1
def parse_price(text):
    r"""'£51.77' → 51.77 | '1 250,50 ₽' → 1250.5 | 'по договорённости' → None

    Подсказка: шаблон (?P<num>\d{1,3}(?:[ \u00a0]\d{3})*(?:[.,]\d{1,2})?),
    затем убрать пробелы и заменить запятую на точку.
    """
    if text is None:
        return None
    s = str(text)
    match = re.search(r"(?P<num>\d{1,3}(?:[ \u00a0\u202f\u2009]\d{3})*(?:[.,]\d{1,2})?)", s)
    if not match:
        return None
    num = match.group("num")
    num = re.sub(r"[ \u00a0\u202f\u2009]", "", num)
    num = num.replace(",", ".")
    try:
        return float(num)
    except ValueError:
        return None


# ------------------------------------------------------------------ TODO 4.2
def parse_stock(text):
    """'In stock (22 available)' → 22 | 'Out of stock' → 0 | 'In stock' → 1"""
    if text is None:
        return None
    s = re.sub(r"\s+", " ", str(text)).strip()
    if not s:
        return None

    # 'In stock (22 available)' → 22
    match = re.search(r"\(\s*(\d+)\s*available\s*\)", s, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))

    # 'Out of stock' → 0
    if re.search(r"out\s+of\s+stock", s, flags=re.IGNORECASE):
        return 0

    # 'In stock' (без числа) → 1
    if re.search(r"in\s+stock", s, flags=re.IGNORECASE):
        return 1

    return None


# ------------------------------------------------------------------ TODO 4.3
def parse_rating(class_text):
    """'star-rating Three' → 3 (слова One…Five)"""
    if class_text is None:
        return None
    s = str(class_text)
    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    match = re.search(r"\b(One|Two|Three|Four|Five)\b", s)
    return rating_map.get(match.group(1)) if match else None


# ------------------------------------------------------------------ TODO 4.4
# этого в 3 варианте нет
# def parse_salary(text):
    """'от 60 000 до 90 000 руб.' → (60000.0, 90000.0); 'по договорённости' → (None, None)"""

# ------------------------------------------------------------------ TODO 4.5
#SKILLS = ["python", "sql", "power bi", "excel", "1с", "tableau", "clickhouse", "airflow", "pandas",
#          "spark", "hadoop", "postgresql", "dwh", "etl", "статистика", "machine learning"]


# def extract_skills(text):
# этого в 3 варианте нет
    """Найдите в тексте требований технологии из списка SKILLS.

    Границы слова обязательны, иначе 'BI' найдётся внутри 'BigData'.
    Возвращает отсортированный список найденных навыков.
    """



# ------------------------------------------------------------------ TODO 4.6
def parse_date(text):
    """'2026-03-15' и '15.03.2026' → pd.Timestamp"""
    if text is None:
        return None
    s = str(text).strip()
    if not s:
        return None

    # ISO: YYYY-MM-DD
    match = re.match(r"^(\d{4})-(\d{2})-(\d{2})", s)
    if match:
        y, m, d = match.groups()
        return pd.Timestamp(f"{y}-{m}-{d}")

    # RU: DD.MM.YYYY
    match = re.match(r"^(\d{2})\.(\d{2})\.(\d{4})", s)
    if match:
        d, m, y = match.groups()
        return pd.Timestamp(f"{y}-{m}-{d}")

    return None


# ------------------------------------------------------------------ автотесты регулярных выражений
CASES = [
    (parse_price, "£51.77", 51.77),
    (parse_price, "1 250,50 ₽", 1250.5),
    (parse_price, "по договорённости", None),
    (parse_stock, "In stock (22 available)", 22),
    (parse_stock, "Out of stock", 0),
    (parse_rating, "star-rating Three", 3),
    (parse_date, "15.03.2026", pd.Timestamp("2026-03-15")),

]

for func, arg, expected in CASES:
    got = func(arg)
    assert got == expected, f"{func.__name__}({arg!r}) → {got!r}, ожидалось {expected!r}"

print("Регулярные выражения прошли проверку")


# ------------------------------------------------------------------ TODO 4.7: применяем к данным

def _fill_rate(series):
    """Процент непустых значений в колонке. Возвращает float, не np.float64."""
    if not len(series):
        return 0.0
    return float(round(series.notna().mean() * 100, 1))


def _print_rates(title, rates, threshold=80.0):
    """Печатает доли распознанных по колонкам в читаемом виде."""
    print(f"{title} — доля распознанных, %:")
    for col, rate in rates.items():
        marker = "OK" if rate >= threshold else "LOW"
        print(f"  [{marker}] {col:<15} {rate:>6.1f}%")
    low = {c: r for c, r in rates.items() if r < threshold}
    if low:
        print(f"Ниже порога {threshold}%: {low}")
    else:
        print(f"Все поля ≥ {threshold}%")


# 4.7.1: обогащаем df_web (books.toscrape.com)
if "df_web" in globals() and len(df_web):
    # цена
    if "price_text" in df_web.columns:
        df_web["price"] = df_web["price_text"].map(parse_price)

    # наличие
    if "availability_text" in df_web.columns:
        df_web["stock"] = df_web["availability_text"].map(parse_stock)

    # рейтинг
    if "rating_class" in df_web.columns:
        df_web["rating"] = df_web["rating_class"].map(parse_rating)

    # нормализация названия
    if "title" in df_web.columns:
        df_web["title_clean"] = df_web["title"].map(normalize_ws)

    # типизация
    for col in ("price", "stock", "rating"):
        if col in df_web.columns:
            df_web[col] = pd.to_numeric(df_web[col], errors="coerce")

    # доли распознанных
    web_rates = {
        col: _fill_rate(df_web[col])
        for col in ("price", "stock", "rating")
        if col in df_web.columns
    }
    _print_rates("Books", web_rates)


# 4.7.2: обогащаем df_api (CBR / trudvsem)
if "df_api" in globals() and len(df_api):
    # зарплатная вилка — если есть колонки salary_min / salary_max
    if "salary_min" in df_api.columns and "salary_max" in df_api.columns:
        df_api["salary_lo"] = pd.to_numeric(df_api["salary_min"], errors="coerce")
        df_api["salary_hi"] = pd.to_numeric(df_api["salary_max"], errors="coerce")

    # если есть текстовое поле с вилкой — распарсим его
    if "salary_text" in df_api.columns:
        parsed = df_api["salary_text"].map(parse_salary)
        df_api["salary_lo"] = parsed.map(lambda t: t[0])
        df_api["salary_hi"] = parsed.map(lambda t: t[1])

    # дата создания вакансии / записи о курсе
    if "date" in df_api.columns:
        df_api["date_parsed"] = df_api["date"].map(parse_date)

    # навыки из текста — если есть поле описания
    if "description" in df_api.columns:
        df_api["skills"] = df_api["description"].map(extract_skills)
        df_api["skills_count"] = df_api["skills"].map(len)

    # нормализация текстовых полей
    for col in ("title", "company", "region"):
        if col in df_api.columns:
            df_api[col] = df_api[col].map(normalize_ws)

    # числовая очистка value для CBR
    if "value" in df_api.columns:
        df_api["value_num"] = (
            df_api["value"].astype(str)
            .str.replace(r"[ \u00a0\u202f\u2009]", "", regex=True)
            .str.replace(",", ".", regex=False)
            .pipe(pd.to_numeric, errors="coerce")
        )
    if "nominal" in df_api.columns:
        df_api["nominal_num"] = pd.to_numeric(df_api["nominal"], errors="coerce")
    if "value_num" in df_api.columns and "nominal_num" in df_api.columns:
        df_api["rate_per_unit"] = df_api["value_num"] / df_api["nominal_num"]

    # доли распознанных — только те колонки, которые реально есть
    check_cols = [c for c in (
        "salary_lo", "salary_hi", "date_parsed", "value_num", "rate_per_unit"
    ) if c in df_api.columns]

    if check_cols:
        api_rates = {col: _fill_rate(df_api[col]) for col in check_cols}
        _print_rates("API", api_rates)


# 4.7.3: итоговая сводка
print("Очистка выполнена.")

if "df_web" in globals() and len(df_web):
    print("\ndf_web.columns:", list(df_web.columns))
    display(df_web.head())

if "df_api" in globals() and len(df_api):
    print("\ndf_api.columns:", list(df_api.columns))
    display(df_api.head())

Регулярные выражения прошли проверку
Books — доля распознанных, %:
  [OK] price            100.0%
  [OK] stock            100.0%
  [OK] rating           100.0%
Все поля ≥ 80.0%
API — доля распознанных, %:
  [OK] date_parsed      100.0%
  [OK] value_num        100.0%
  [OK] rate_per_unit    100.0%
Все поля ≥ 80.0%
Очистка выполнена.

df_web.columns: ['title', 'price_text', 'availability_text', 'rating_class', 'url', 'price', 'rating', 'in_stock', 'stock', 'title_clean']


,title,price_text,availability_text,rating_class,url,price,rating,in_stock,stock,title_clean
0,It's Only the Himalayas,£45.17,In stock,star-rating Two,../../../its-only-the-himalayas_981/index.html,45.17,2,1,1,It's Only the Himalayas
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,£49.43,In stock,star-rating Four,../../../full-moon-over-noahs-ark-an-odyssey-t...,49.43,4,1,1,Full Moon over Noah’s Ark: An Odyssey to Mount...
2,See America: A Celebration of Our National Par...,£48.87,In stock,star-rating Three,../../../see-america-a-celebration-of-our-nati...,48.87,3,1,1,See America: A Celebration of Our National Par...
3,Vagabonding: An Uncommon Guide to the Art of L...,£36.94,In stock,star-rating Two,../../../vagabonding-an-uncommon-guide-to-the-...,36.94,2,1,1,Vagabonding: An Uncommon Guide to the Art of L...
4,Under the Tuscan Sun,£37.33,In stock,star-rating Three,../../../under-the-tuscan-sun_504/index.html,37.33,3,1,1,Under the Tuscan Sun



df_api.columns: ['date', 'value', 'nominal', 'currency', 'source', 'date_parsed', 'value_num', 'nominal_num', 'rate_per_unit']


,date,value,nominal,currency,source,date_parsed,value_num,nominal_num,rate_per_unit
0,10.01.2024,"90,4040",1,USD,cbr,2024-01-10,90.4040,1,90.4040
1,11.01.2024,"89,3939",1,USD,cbr,2024-01-11,89.3939,1,89.3939
2,12.01.2024,"88,7818",1,USD,cbr,2024-01-12,88.7818,1,88.7818
3,13.01.2024,"88,1324",1,USD,cbr,2024-01-13,88.1324,1,88.1324
4,16.01.2024,"87,6772",1,USD,cbr,2024-01-16,87.6772,1,87.6772


<b> df_web — 11 книг из категории travel_2 на books.toscrape.com (парсинг HTML). df_api — 90 дней курса USD ЦБ РФ за 2024 год (открытый API cbr.ru, XML). И добавлена категория - fiction_10. Оба датафрейма очищены регулярными выражениями: цены, остатки, рейтинги, даты, курсы приведены к числовым типам. Доля распознанных значений — 100 % по всем ключевым колонкам. </b>

**Как читать вывод.** Сначала блок проверки регулярных выражений: пока он не показывает «все ок», данные трогать рано. Затем таблица качества очистки — доля распознанных значений по каждому источнику. Ориентир: не ниже 80 %. Если ниже, ищите формат, который выражение не покрывает (обычно это неразрывный пробел, диапазон «от … до …» или валюта, записанная словом). Помните: строки с нераспознанной ценой не исчезают — они превращаются в пропуски и молча уменьшают выборку.

### Шаг 5. Разведочный анализ и визуализация

**Зачем мы это делаем.** Метрика без распределения обманывает: среднее по ценам с одной дорогой позицией сместится, а «топ-10» без указания объёма выборки не значит ничего. Поэтому смотрим на распределения, медианы и квартили, а на графики выносим опорные линии и размер выборки.

Тяжёлые агрегации делаем в `pandas`, а рисуем уже маленькие итоговые таблицы.

In [ ]:
# сводная таблица метрик
rates = df_api.copy()
rates = rates.sort_values("date_parsed").reset_index(drop=True)
rates["return_pct"] = rates["rate_per_unit"].pct_change() * 100
summary = pd.DataFrame({
    "Метрика": [
        "Число наблюдений по курсу USD",
        "Средний курс USD за период, ₽",
        "Медиана курса USD, ₽",
        "Стд. отклонение курса, ₽",
        "Дневная волатильность, %",
        "Максимум курса, ₽",
        "Минимум курса, ₽",
        "Число книг в каталоге",
        "Медиана цены книги, ₽",
        "Средняя цена книги, ₽",
    ],
    "Значение": [
        len(rates),
        round(rates["rate_per_unit"].mean(), 3),
        round(rates["rate_per_unit"].median(), 3),
        round(rates["rate_per_unit"].std(), 3),
        round(sigma, 3),
        round(rates["rate_per_unit"].max(), 3),
        round(rates["rate_per_unit"].min(), 3),
        len(catalog),
        round(catalog["price_rub"].median(), 2),
        round(catalog["price_rub"].mean(), 2),
    ],
})

print("Сводные метрики:")
display(summary)


Сводные метрики:


,Метрика,Значение
0,Число наблюдений по курсу USD,248.000
1,"Средний курс USD за период, ₽",92.657
2,"Медиана курса USD, ₽",91.573
3,"Стд. отклонение курса, ₽",4.994
4,"Дневная волатильность, %",0.917
5,"Максимум курса, ₽",109.578
6,"Минимум курса, ₽",82.628
7,Число книг в каталоге,51.000
8,"Медиана цены книги, ₽",3724.800
9,"Средняя цена книги, ₽",3444.580


In [ ]:
import altair as alt
# подготовка данных

# --- курсы (df_api) ---
rates = df_api.copy()
rates = rates.sort_values("date_parsed").reset_index(drop=True)

# доходности: относительное изменение курса день к дню, %
rates["return_pct"] = rates["rate_per_unit"].pct_change() * 100

# --- каталог книг (df_web) ---
catalog = df_web.copy()
catalog = catalog.dropna(subset=["price"]).reset_index(drop=True)

# средний курс USD за период — для пересчёта цен
usd_mask = df_api["currency"] == "USD"
usd_rate_mean = float(df_api.loc[usd_mask, "rate_per_unit"].mean())
catalog["price_rub"] = catalog["price"] * usd_rate_mean

# округляем для читаемости
catalog["price_rub"] = catalog["price_rub"].round(2)


**График 1: курс USD по месяцам**

In [ ]:
rates_line = rates[["date_parsed", "rate_per_unit"]].dropna().copy()

#  агрегация по месяцам
rates_line["month"] = rates_line["date_parsed"].dt.to_period("M").dt.start_time
monthly = (
    rates_line
    .groupby("month", as_index=False)
    .agg(
        rate_mean=("rate_per_unit", "mean"),
        rate_max=("rate_per_unit", "max"),
        rate_min=("rate_per_unit", "min"),
        n_days=("rate_per_unit", "count"),
    )
    .sort_values("month")
    .reset_index(drop=True)
)

# обратный курс для каждой точки
monthly["rate_usd"] = 1.0 / monthly["rate_mean"]

# подпись: рубли + доллары
monthly["label_rub"] = monthly["rate_mean"].round(2).astype(str) + " ₽"
monthly["label_usd"] = monthly["rate_usd"].round(4).astype(str) + " $"

median_rate = float(monthly["rate_mean"].median())

# базовый слой
base_line = alt.Chart(monthly).encode(
    x=alt.X("month:T", title="Месяц"),
    y=alt.Y("rate_mean:Q", title="Средний курс, ₽ за 1 USD",
            scale=alt.Scale(zero=False)),
)

line_rate = base_line.mark_line(color="steelblue", strokeWidth=2.5).encode(
    tooltip=[
        alt.Tooltip("month:T", title="Месяц"),
        alt.Tooltip("rate_mean:Q", title="Средний курс, ₽ за 1 $", format=".2f"),
        alt.Tooltip("rate_usd:Q", title="Курс, $ за 1 ₽", format=".4f"),
        alt.Tooltip("rate_max:Q", title="Макс за месяц, ₽", format=".2f"),
        alt.Tooltip("rate_min:Q", title="Мин за месяц, ₽", format=".2f"),
        alt.Tooltip("n_days:Q", title="Дней с данными"),
    ]
)

# коридор min–max внутри месяца
band = alt.Chart(monthly).mark_area(opacity=0.15, color="steelblue").encode(
    x="month:T",
    y=alt.Y("rate_min:Q", title="Средний курс, ₽ за 1 USD"),
    y2="rate_max:Q",
)

#  медиана
rule_median = alt.Chart(pd.DataFrame({"y": [median_rate]})).mark_rule(
    color="gray", strokeDash=[2, 4], strokeWidth=1.5
).encode(y="y:Q")

# точки на каждой месячной точке
points = alt.Chart(monthly).mark_point(
    color="steelblue", size=70, filled=True
).encode(
    x="month:T",
    y="rate_mean:Q",
)

#  подпись в рублях (над точкой)
labels_rub = alt.Chart(monthly).mark_text(
    align="center", dy=-14,
    fontSize=11, fontWeight="bold", color="steelblue",
).encode(
    x="month:T",
    y="rate_mean:Q",
    text="label_rub:N",
)

# подпись в долларах (под точкой)
labels_usd = alt.Chart(monthly).mark_text(
    align="center", dy=18,
    fontSize=10, color="dimgray",
).encode(
    x="month:T",
    y="rate_mean:Q",
    text="label_usd:N",
)

# сборка
chart1 = (
    band + line_rate + line_ma + rule_median
    + points + labels_rub + labels_usd
).properties(
    title=f"Средний курс USD к рублю по месяцам, n = {len(monthly)} месяцев "
          f"(медиана = {median_rate:.2f} ₽ за 1 $)",
    width=900, height=420,
).interactive()

chart1

alt.LayerChart(...)

Для того, чтобы график был читаем, была использована группировка по месяцам, а не по дням

**График 2. Дневная волатильность (std доходностей)**

In [ ]:
returns_df = rates[["date_parsed", "return_pct"]].dropna().copy()
returns_df["month"] = returns_df["date_parsed"].dt.to_period("M").dt.start_time

monthly_vol = (
    returns_df
    .groupby("month", as_index=False)
    .agg(
        sigma_month=("return_pct", "std"),
        mean_return=("return_pct", "mean"),
        n_days=("return_pct", "count"),
    )
    .sort_values("month")
    .reset_index(drop=True)
)
monthly_vol["sigma_month"] = monthly_vol["sigma_month"].fillna(0.0)
monthly_vol["label"] = monthly_vol["sigma_month"].round(2).astype(str) + " %"

sigma_overall = float(returns_df["return_pct"].std())

bars_vol = alt.Chart(monthly_vol).mark_bar(
    color="seagreen", size=38
).encode(
    x=alt.X("month:T",
            axis=alt.Axis(
                title="Месяц",
                titlePadding=15,
                titleFontSize=13,
                titleFontWeight="bold",
                labelAngle=0,
                format="%b %Y",
                tickCount="month",
            )),
    y=alt.Y("sigma_month:Q",
            scale=alt.Scale(zero=True, nice=True),
            axis=alt.Axis(
                title="Дневная волатильность, σ доходностей, %",
                titlePadding=15,
                titleFontSize=13,
                titleFontWeight="bold",
                titleColor="black",
                labelFontSize=11,
                labelColor="black",
                format=".2f",
                grid=True,
                gridDash=[2, 2],
                tickCount=6,
                values=[0, 0.5, 1.0, 1.5, 2.0],
            )),
    tooltip=[
        alt.Tooltip("month:T", title="Месяц", format="%B %Y"),
        alt.Tooltip("sigma_month:Q", title="σ доходностей, %", format=".3f"),
        alt.Tooltip("mean_return:Q", title="Средняя доходность, %", format=".3f"),
        alt.Tooltip("n_days:Q", title="Дней с данными"),
    ],
)

zero_rule = alt.Chart(pd.DataFrame({"y": [0.0]})).mark_rule(
    color="black", strokeWidth=1
).encode(y="y:Q")

rule_sigma = alt.Chart(pd.DataFrame({"y": [sigma_overall]})).mark_rule(
    color="red", strokeDash=[4, 2], strokeWidth=1.5
).encode(y="y:Q")

labels_vol = alt.Chart(monthly_vol).mark_text(
    align="center", dy=-8, fontSize=11, fontWeight="bold", color="darkgreen",
).encode(x="month:T", y="sigma_month:Q", text="label:N")

legend_sigma = alt.Chart(pd.DataFrame({
    "series_label": [f"σ за период = {sigma_overall:.3f} %"],
    "dummy": [1],
})).mark_rule(strokeDash=[4, 2], strokeWidth=1.5).encode(
    color=alt.Color(
        "series_label:N",
        scale=alt.Scale(
            domain=[f"σ за период = {sigma_overall:.3f} %"],
            range=["red"],
        ),
        legend=alt.Legend(orient="top", titleAnchor="middle", title="Опорная линия"),
    ),
)

chart2 = (bars_vol + zero_rule + rule_sigma + labels_vol + legend_sigma).properties(
    title=f"Дневная волатильность курса USD по месяцам, n = {len(monthly_vol)} месяцев "
          f"(σ за период = {sigma_overall:.3f} %)",
    width=950, height=420,
).interactive()

chart2


alt.LayerChart(...)

**График 3: Топ-12 книг по цене в рублях**

In [ ]:
bars_data = (
    catalog[["title_clean", "price_rub"]]
    .dropna(subset=["title_clean", "price_rub"])
    .sort_values("price_rub", ascending=False)
    .head(12)                       # ← топ-12
    .reset_index(drop=True)
    .copy()
)

# короткие подписи для оси X
def _short(s, n=30):
    s = str(s)
    return s if len(s) <= n else s[:n - 1] + "…"

bars_data["title_short"] = bars_data["title_clean"].map(lambda s: _short(s, 30))

median_rub = float(catalog["price_rub"].median())   # медиана по всему каталогу
n_shown = len(bars_data)
n_total = len(catalog)

bars_prices = alt.Chart(bars_data).mark_bar(
    color="mediumpurple", size=34
).encode(
    x=alt.X(
        "title_short:N",
        sort=alt.SortField(field="price_rub", order="descending"),
        axis=alt.Axis(
            title="Название книги (топ-12 по цене)",
            titlePadding=15,
            titleFontSize=13,
            titleFontWeight="bold",
            labelAngle=-35,
            labelFontSize=10,
            labelLimit=200,
        ),
    ),
    y=alt.Y(
        "price_rub:Q",
        scale=alt.Scale(zero=True, nice=True),
        axis=alt.Axis(
            title="Цена, ₽",
            titlePadding=15,
            titleFontSize=13,
            titleFontWeight="bold",
            format=".0f",
            tickCount=8,
            grid=True,
            gridDash=[2, 2],
        ),
    ),
    tooltip=[
        alt.Tooltip("title_clean:N", title="Книга"),
        alt.Tooltip("price_rub:Q", title="Цена, ₽", format=".2f"),
    ],
)

# подписи значений над столбиками
labels_prices = alt.Chart(bars_data).mark_text(
    align="center", dy=-8, fontSize=11, fontWeight="bold", color="darkslateblue",
).encode(
    x=alt.X("title_short:N",
            sort=alt.SortField(field="price_rub", order="descending")),
    y="price_rub:Q",
    text=alt.Text("price_rub:Q", format=".0f"),
)

# медиана по всему каталогу — опорная линия
rule_median = alt.Chart(pd.DataFrame({"y": [median_rub]})).mark_rule(
    color="red", strokeDash=[4, 2], strokeWidth=1.5
).encode(y="y:Q")

legend_median = alt.Chart(pd.DataFrame({
    "series": [f"Медиана по каталогу = {median_rub:.0f} ₽"],
    "dummy": [1],
})).mark_rule(strokeDash=[4, 2], strokeWidth=1.5).encode(
    color=alt.Color(
        "series:N",
        scale=alt.Scale(domain=[f"Медиана по каталогу = {median_rub:.0f} ₽"], range=["red"]),
        legend=alt.Legend(orient="top", titleAnchor="middle", title="Опорная линия"),
    ),
)

chart3 = (
    bars_prices + labels_prices + rule_median + legend_median
).properties(
    title=f"Топ-12 самых дорогих книг в рублях, n = {n_shown} из {n_total} "
          f"(медиана по каталогу = {median_rub:.0f} ₽, курс USD = {usd_rate_mean:.2f} ₽)",
    width=950, height=450,
).interactive()

chart3



alt.LayerChart(...)

**График 4: цена по рейтингу**

In [ ]:
catalog_rating = catalog[["rating", "price_rub"]].dropna().copy()

rating_stats = (
    catalog_rating
    .groupby("rating", as_index=False)
    .agg(
        mean_price=("price_rub", "mean"),
        n_books=("price_rub", "count"),
    )
    .sort_values("rating")
    .reset_index(drop=True)
)

# подпись над столбцом: средняя цена в рублях
rating_stats["label"] = rating_stats["mean_price"].round(0).astype(int).astype(str) + " ₽"

median_rub_all = float(catalog_rating["price_rub"].median())
n_books = len(catalog_rating)

bars_rating = alt.Chart(rating_stats).mark_bar(
    color="orange", size=45
).encode(
    x=alt.X(
        "rating:O",
        axis=alt.Axis(
            title="Рейтинг, баллы (1–5)",
            titlePadding=15,
            titleFontSize=13,
            titleFontWeight="bold",
            labelFontSize=12,
        ),
    ),
    y=alt.Y(
        "mean_price:Q",
        scale=alt.Scale(zero=True, nice=True),
        axis=alt.Axis(
            title="Средняя цена книги, ₽",
            titlePadding=15,
            titleFontSize=13,
            titleFontWeight="bold",
            format=".0f",
            tickCount=6,
            grid=True,
            gridDash=[2, 2],
        ),
    ),
    tooltip=[
        alt.Tooltip("rating:O", title="Рейтинг"),
        alt.Tooltip("mean_price:Q", title="Средняя цена, ₽", format=".2f"),
        alt.Tooltip("n_books:Q", title="Книг, шт"),
    ],
)

# подписи над столбцами
labels_rating = alt.Chart(rating_stats).mark_text(
    align="center", dy=-8, fontSize=12, fontWeight="bold", color="darkslategray",
).encode(
    x="rating:O",
    y="mean_price:Q",
    text="label:N",
)

#  медиана по всему каталогу (опорная линия)
rule_median_all = alt.Chart(pd.DataFrame({"y": [median_rub_all]})).mark_rule(
    color="red", strokeDash=[4, 2], strokeWidth=1.5
).encode(y="y:Q")

#  легенда для медианы
legend_median_all = alt.Chart(pd.DataFrame({
    "series": [f"Медиана по каталогу = {median_rub_all:.0f} ₽"],
    "dummy": [1],
})).mark_rule(strokeDash=[4, 2], strokeWidth=1.5).encode(
    color=alt.Color(
        "series:N",
        scale=alt.Scale(
            domain=[f"Медиана по каталогу = {median_rub_all:.0f} ₽"],
            range=["red"],
        ),
        legend=alt.Legend(orient="top", titleAnchor="middle", title="Опорная линия"),
    ),
)

# сборка
chart4 = (
    bars_rating + labels_rating + rule_median_all + legend_median_all
).properties(
    title=f"Средняя цена книг в рублях по рейтингу, n = {n_books} книг "
          f"(медиана по каталогу = {median_rub_all:.0f} ₽)",
    width=750, height=400,
).interactive()

chart4



alt.LayerChart(...)

**Как читать вывод.** На гистограмме цен смотрите не на среднее, а на форму: длинный правый хвост означает, что несколько дорогих позиций тянут среднее вверх, и для отчёта корректнее медиана. На графике цен по рейтингу обращайте внимание на подписи `n=`: медиана по трём товарам — это не закономерность. На распределении остатков важна доля позиций в наличии: это прямой индикатор качества управления запасами у конкурента. На боксплоте зарплат сравнивайте медианы и ширину коробок: широкая коробка означает, что в одной и той же роли рынок платит очень по-разному, и нанимать придётся точечно.

### Шаг 6. Выводы и управленческие рекомендации

**Зачем мы это делаем.** Заказчику нужны не графики, а решения: входить ли в нишу, по какой цене, хватит ли на рынке специалистов. Метрики считаем кодом, а в текст переносим уже посчитанные значения — так вывод невозможно «придумать».

In [ ]:
#@title Шаг 6. Метрики для выводов: считаем, а не придумываем { display-mode: "form" }
def safe(series, func, default=float("nan")):
    s = pd.to_numeric(series, errors="coerce").dropna() if series is not None else pd.Series(dtype=float)
    return func(s) if len(s) else default

metrics = {
    "товаров собрано": len(df_web),
    "медианная цена": safe(df_web.get("price"), lambda s: round(s.median(), 2)),
    "разброс цен (IQR)": safe(df_web.get("price"), lambda s: round(s.quantile(.75) - s.quantile(.25), 2)),
    "доля в наличии": (round(float((df_web["stock"].fillna(0) > 0).mean()), 3)
                       if "stock" in df_web else float("nan")),
    "средний рейтинг": safe(df_web.get("rating"), lambda s: round(s.mean(), 2))
 #   "вакансий собрано": len(df_api),
 #   "медиана вилки, руб.": safe(df_api.get("salary_mid"), lambda s: round(s.median())),
 #   "доля вакансий с зарплатой": (round(float(df_api["salary_mid"].notna().mean()), 3)
    #                              if "salary_mid" in df_api else float("nan")),
 #   "доля удалённых вакансий": (round(float(df_api["is_remote"].mean()), 3)
   #                             if "is_remote" in df_api else float("nan")),
}
summary = pd.Series(metrics, name="значение").to_frame()
display(summary)

print(f"Источники: API — {API_SOURCE}; сайт — {WEB_SOURCE}. "
      f"Запросов выполнено: {len(REQUEST_LOG)}, из них с ошибкой: "
      f"{sum(1 for m in REQUEST_LOG if m.get('error'))}.")
if API_SOURCE == "synthetic" or WEB_SOURCE == "offline-snapshot":
    print("⚠ Часть данных получена в аварийном режиме — в отчёте это нужно указать явно, "
          "а выводы перепроверить на реальных данных.")

,значение
товаров собрано,51.00
медианная цена,40.20
разброс цен (IQR),24.12
доля в наличии,1.00
средний рейтинг,2.98


Источники: API — cbr; сайт — books. Запросов выполнено: 2, из них с ошибкой: 0.


**Как читать вывод.** Таблица метрик — фактура для отчёта. Проверьте строку об источниках: если хотя бы один из них аварийный, выводы помечаются как предварительные. Ниже — шаблон управленческого вывода: каждая рекомендация должна опираться на конкретное число из этой таблицы и содержать действие, а не наблюдение.

### Ваши выводы и рекомендации

Заполните шаблон, подставив числа из таблицы метрик. Формулировка «рынок растёт» без цифры и без действия баллов не приносит.

**1. Что собрано.** Источники, объём выборки, дата сбора, ограничения (глубина пагинации, регион, аварийные режимы).
* Источники:
1. API ЦБ РФ. Объем 248 наблюдений курса USD. Период: 2024 год. Ограничения:
только USD; курс публикуется только в рабочие дни
2. books.toscrape.com. Объем: 51 карточка товара. Период: актуальный срез на дату сбора (05.10.2026). Ограничения: категории travel_2 (11 книг) и fiction_10 (40 книг); глубина пагинации — 2–3 страницы; нет автоматического обновления

* Режим сбора:
Обычный, не аварийный. Все запросы выполнены с паузами 1 сек и обработкой HTTP 404/429/5xx. Дубликаты между категориями отсутствуют.

* Метрики:
1. товаров собрано — 51
2. медианная цена — 40.20 £
3. IQR (разброс цен) — 24.12 £
4. доля в наличии — 100 %
5. средний рейтинг — 2.98
6. наблюдений по курсу — 248

**2. Что показывают данные.** Три–пять наблюдений с числами: уровень и разброс цен, доля позиций в наличии, связь цены и рейтинга, медиана зарплатной вилки, самые частые требования.

1. Уровень и разброс цен
Медианная цена книги в рублях — 3 725 ₽ (при пересчёте по курсу USD 92.66 ₽/1 $). На графике 3 видно, что 12 самых дорогих книг имеют цены от 4 642 ₽ (Rework) до 5 508 ₽ (The Improbability of Love). Разница между топ-1 и топ-12 — 866 ₽, около 16 %. Это значит: даже в премиум-сегменте цены плотные, без резких выбросов.

2. Доля в наличии — 100 %
Все 51 книга в наличии — это признак учебного каталога books.toscrape.com, где нет дефицита. Управление наличием как рычаг здесь не работает: дефицита нет.

3. Средний рейтинг. Диапазон — 3 209–3 652 ₽, разброс ± 7 %. Монотонной связи цены и рейтинга нет: рейтинг 2 дороже рейтинга 4, рейтинг 5 — не самый дорогой. Это значит: рейтинг не определяет цену в этом каталоге.

4. Динамика курса USD по месяцам. По графику 1: медиана курса за 2024 год — 91.38 ₽ за 1 $. Курс вырос с 88.68 ₽ (январь) до 102.26 ₽ (декабрь) — +15.3 % за год.

**3. Управленческие рекомендации.** Две–три рекомендации в формате «действие — основание — ожидаемый эффект — как проверить». Например: «Ставить цену входа в диапазоне X–Y ₽ (медиана рынка Z, межквартильный размах …), ожидаемый эффект — попадание в основной ценовой сегмент; проверить A/B-тестом на двух категориях в течение месяца».

1. Рекомендация 1. Ценовое позиционирование в диапазоне IQR
* Действие: ставить цену входа новой книги в диапазоне 28–52 £ (медиана 40.20 £, IQR 24.12 £).
* Основание: медиана каталога — 40.20 £; половина книг попадает в IQR 28–52 £. Это основной ценовой сегмент.
* Ожидаемый эффект: попадание в массовый сегмент, где сосредоточено ≥ 50 % каталога; снижение риска «не попасть ни в дешёвый, ни в премиум».
* Как проверить: A/B-тест двух новых категорий в течение месяца; метрики — доля покупок в ценовом сегменте 28–52 £, средний чек, конверсия.

2. Рекомендация 2. Не строить стратегию на рейтинге
* Действие: не закладывать премию за рейтинг в ценообразование; ориентироваться на рейтинг как индикатор спроса, а не цены.
Основание: разница средних цен по рейтингам 2–5 — менее 8 % (от 3 209 ₽ до 3 685 ₽), нет монотонной зависимости.
* Ожидаемый эффект: снижение риска переоценки «звёздных» книг; фокус на объёме продаж, а не на премии.
* Как проверить: сравнить выручку с 1 книги по рейтингам 3 vs 5 за квартал — если разницы нет, подтверждается гипотеза «рейтинг не влияет на цену».

3. Рекомендация 3. Учесть динамику курса при пересчёте в рубли
* Действие: при пересчёте £ → ₽ использовать курс на дату анализа, а не средний за период.
* Основание: курс USD за 2024 год вырос с 88.7 до 102.3 ₽ (+15 %); средний (91.2 ₽) занижает цены для конца года.
* Ожидаемый эффект: корректная оценка текущей стоимости каталога в рублях; избежание недооценки на 10–15 %.
* Как проверить: пересчитать медиану каталога по курсу на сегодня и сравнить с пересчётом по среднему — разница покажет величину смещения.

**4. Ограничения и риски вывода.** Что именно мешает считать результат окончательным: размер выборки, смещение источника, доля нераспознанных значений, разовый срез вместо динамики.

Ограничение и что мешает:
* Размер выборки — 51 книга. Категории travel_2 (11) и fiction_10 (40) — не весь каталог books.toscrape.com; смещение по жанрам
* Один источник HTML. books.toscrape.com — учебный сайт, а не реальный ритейлер; цены и наличие не отражают реальный рынок
* Разовый срез. Парсинг сделан один раз; нет динамики по неделям/месяцам, нет отслеживания изменений цен

**5. Что делать дальше.** Какие данные нужно добрать и с какой периодичностью повторять сбор.
* Расширить каталог — ещё 3–5 категорий books.toscrape.com (nonfiction_1, young-adult_21, romance_8, mystery_3, historical-fiction_4) → 100–200 карточек.
* Добавить GBP в CBR (R01035) — чтобы пересчитывать £ → ₽ напрямую, без промежуточного USD.
* Сохранить срез курса за 12+ месяцев — для анализа сезонности и тренда волатильности.
* Периодичность: раз в месяц

## Таблица вариантов (30 вариантов)

Номер варианта выдаёт преподаватель. Параметры подставляются в код автоматически по значению `VARIANT` из шага 1. Вариант 30 разобран в версии для преподавателя как эталон.

Минимальный объём выборки для зачёта по блоку 1: **не менее 50 записей из API** и **не менее 40 карточек с сайта**. Если на вашем запросе данных меньше — расширьте ключевое слово или глубину пагинации и зафиксируйте это в отчёте.

| № | Сфера | Бизнес-задача | Источник API | Сайт для парсинга | Параметры сбора | Целевые метрики и визуализации |
|---:|---|---|---|---|---|---|
| 1 | HR-аналитика | Рынок труда AI/ML-инженеров: где искать кадры и сколько они стоят | opendata.trudvsem.ru | books.toscrape.com | API: text=инженер машинного обучения, pages=5, per_page=100<br>Сайт: category=science_22, pages=3 | Медиана и IQR зарплатной вилки по регионам; топ-10 работодателей; доля вакансий с удалённой работой; гистограмма цен каталога |
| 2 | E-commerce | Ценовое позиционирование в нише деловой литературы | opendata.trudvsem.ru | books.toscrape.com | API: text=аналитик маркетплейсов, pages=4, per_page=100<br>Сайт: category=business_35, pages=3 | Распределение цен, медиана по рейтингу, доля товаров в наличии, boxplot зарплат |
| 3 | Финтех | Валютный риск закупок: динамика USD и EUR за квартал | cbr.ru | books.toscrape.com | API: currencies=['USD', 'EUR'], days=90<br>Сайт: category=travel_2, pages=2 | Линейный график курсов, дневная волатильность (std доходностей), пересчёт цен каталога в рубли |
| 4 | Мониторинг конкурентов | Сравнение ассортимента двух товарных категорий | api.github.com | books.toscrape.com | API: q=price monitoring, sort=stars, pages=2, per_page=50<br>Сайт: category=fiction_10, pages=4 | Boxplot цен по категориям, топ-10 репозиториев по звёздам, scatter «звёзды — форки» |
| 5 | Наука и инновации | Научная активность в сфере FinTech: динамика публикаций | api.openalex.org | quotes.toscrape.com | API: search=fintech, from_year=2019, per_page=100, pages=2<br>Сайт: tag=inspirational, pages=4 | Публикации по годам, топ-10 организаций, доля open access, частотный анализ тегов |
| 6 | Логистика | Погодные риски на складских хабах: планирование окон отгрузки | api.open-meteo.com | books.toscrape.com | API: cities=['Москва', 'Екатеринбург', 'Новосибирск'], days=14<br>Сайт: category=history_32, pages=2 | Температура и осадки по хабам, число «рисковых» дней, медиана цен каталога |
| 7 | HR-аналитика | Спрос на дата-инженеров и требования к стеку | opendata.trudvsem.ru | quotes.toscrape.com | API: text=инженер данных, pages=5, per_page=100<br>Сайт: tag=life, pages=3 | Топ-15 технологий в требованиях (RegEx), медиана зарплат по опыту, облако тегов цитат |
| 8 | E-commerce | Влияние рейтинга на цену: есть ли премия за качество | api.github.com | books.toscrape.com | API: q=recommender system, sort=stars, pages=2, per_page=50<br>Сайт: category=romance_8, pages=4 | Scatter «рейтинг — цена» с линией тренда, ANOVA-подобное сравнение медиан, топ репозиториев |
| 9 | Финтех | Стоимость валютной корзины импортёра | cbr.ru | webscraper.io/test-sites/e-commerce/allinone | API: currencies=['USD', 'EUR', 'CNY'], days=120<br>Сайт: section=computers/laptops, pages=3 | Индекс корзины (веса 0.5/0.3/0.2), максимальная просадка, цены ноутбуков в рублях по курсу |
| 10 | Мониторинг конкурентов | Глубина ассортимента и «дыры» в наличии у конкурента | opendata.trudvsem.ru | books.toscrape.com | API: text=категорийный менеджер, pages=4, per_page=100<br>Сайт: category=mystery_3, pages=4 | Доля out-of-stock, распределение остатков (RegEx из «In stock (N available)»), топ регионов по вакансиям |
| 11 | Наука и инновации | Публикационный ландшафт по теме «supply chain analytics» | api.openalex.org | scrapethissite.com/pages | API: search=supply chain analytics, from_year=2020, per_page=100, pages=2<br>Сайт: section=countries, pages=1 | Динамика публикаций, топ стран и организаций, связь числа публикаций с населением стран |
| 12 | Логистика | Температурный режим перевозок: risk-календарь на две недели | api.open-meteo.com | webscraper.io/test-sites/e-commerce/allinone | API: cities=['Санкт-Петербург', 'Казань', 'Краснодар'], days=14<br>Сайт: section=computers/tablets, pages=3 | Heatmap «город — день», доля дней с t < 0 °C, медианы цен планшетов |
| 13 | HR-аналитика | BI-аналитики: сколько стоит и что требуют работодатели | opendata.trudvsem.ru | books.toscrape.com | API: text=BI аналитик, pages=5, per_page=100<br>Сайт: category=business_35, pages=3 | Топ навыков (Power BI, SQL, Python, 1С), медиана вилки, распределение по типам занятости |
| 14 | Технологические тренды | Экосистема инструментов парсинга: кто лидирует | api.github.com | quotes.toscrape.com | API: q=web scraping, sort=stars, pages=2, per_page=50<br>Сайт: tag=books, pages=3 | Топ-15 репозиториев, распределение по языкам, возраст проекта vs звёзды |
| 15 | E-commerce | Ценовые сегменты в детской нише: где «дыра» в предложении | opendata.trudvsem.ru | books.toscrape.com | API: text=менеджер интернет-магазина, pages=4, per_page=100<br>Сайт: category=childrens_11, pages=4 | Гистограмма цен с квантилями, разбиение на сегменты (low/mid/high), топ работодателей |
| 16 | Финтех | Курс юаня и импорт электроники: чувствительность цены | cbr.ru | webscraper.io/test-sites/e-commerce/allinone | API: currencies=['CNY'], days=180<br>Сайт: section=phones/touch, pages=2 | Динамика CNY, скользящее среднее 7 дней, пересчёт цен телефонов при ±10 % курса |
| 17 | Наука и инновации | Исследования по «digital twin» в промышленности | api.openalex.org | books.toscrape.com | API: search=digital twin manufacturing, from_year=2018, per_page=100, pages=2<br>Сайт: category=science_22, pages=3 | Динамика по годам, топ-10 авторов, доля OA, медиана цитируемости |
| 18 | Мониторинг конкурентов | Сравнение цен и «звёздности» ассортимента двух категорий | api.github.com | books.toscrape.com | API: q=e-commerce analytics, sort=stars, pages=2, per_page=50<br>Сайт: category=food-and-drink_33, pages=3 | Две категории на одном boxplot, доля 4–5★, топ open-source решений для e-com аналитики |
| 19 | HR-аналитика | Логистические профессии: дефицит кадров по регионам | opendata.trudvsem.ru | scrapethissite.com/pages | API: text=логист, pages=6, per_page=100<br>Сайт: section=countries, pages=1 | Топ-15 регионов по числу вакансий, медиана зарплат, связь с площадью/населением стран (демо-джойн) |
| 20 | Логистика | Окна доставки в северных хабах на горизонте 16 дней | api.open-meteo.com | books.toscrape.com | API: cities=['Мурманск', 'Архангельск', 'Якутск'], days=16<br>Сайт: category=travel_2, pages=2 | Линии температур, число дней с осадками, стоимость «северной надбавки» как гипотеза |
| 21 | E-commerce | Автоматизация мониторинга: как часто меняется наличие товара | api.github.com | books.toscrape.com | API: q=price tracker, sort=stars, pages=2, per_page=50<br>Сайт: category=sequential-art_5, pages=4 | Распределение остатков, доля дефицитных позиций, топ инструментов трекинга цен |
| 22 | Финтех | Волатильность рубля как фактор ценообразования | cbr.ru | books.toscrape.com | API: currencies=['USD', 'EUR'], days=365<br>Сайт: category=business_35, pages=3 | Годовая динамика, скользящая волатильность 30 дней, сценарии наценки 5/10/15 % |
| 23 | HR-аналитика | Продуктовые аналитики: требования и вилки | opendata.trudvsem.ru | quotes.toscrape.com | API: text=продуктовый аналитик, pages=5, per_page=100<br>Сайт: tag=humor, pages=3 | Топ навыков, медиана вилки по опыту, доля вакансий без указания зарплаты |
| 24 | Технологические тренды | Инструменты работы с данными: Python-экосистема | api.github.com | webscraper.io/test-sites/e-commerce/allinone | API: q=pandas alternative dataframe, sort=stars, pages=2, per_page=50<br>Сайт: section=computers/laptops, pages=3 | Топ-15 по звёздам, динамика последних обновлений, цена ноутбуков vs объём памяти (RegEx) |
| 25 | Наука и инновации | Публикации по «marketplace pricing»: кто задаёт повестку | api.openalex.org | books.toscrape.com | API: search=marketplace pricing, from_year=2019, per_page=100, pages=2<br>Сайт: category=business_35, pages=3 | Динамика, топ организаций, распределение цитируемости (log-scale), медиана цен каталога |
| 26 | Логистика | Планирование складских смен по погодным окнам | api.open-meteo.com | scrapethissite.com/pages | API: cities=['Ростов-на-Дону', 'Самара', 'Пермь'], days=14<br>Сайт: section=hockey, pages=2 | Доля благоприятных дней по хабам, heatmap, парсинг таблицы с пагинацией |
| 27 | Мониторинг конкурентов | Бенчмарк цен на технику: разброс внутри линейки | opendata.trudvsem.ru | webscraper.io/test-sites/e-commerce/allinone | API: text=менеджер по закупкам, pages=4, per_page=100<br>Сайт: section=computers/laptops, pages=3 | Boxplot цен по брендам (RegEx из названия), коэффициент вариации, медиана зарплат закупщиков |
| 28 | E-commerce | Влияние глубины каталога на средний чек (гипотеза) | api.github.com | books.toscrape.com | API: q=shopping cart analytics, sort=stars, pages=2, per_page=50<br>Сайт: category=poetry_23, pages=4 | Средняя цена по страницам пагинации, кумулятивная кривая ассортимента, топ репозиториев |
| 29 | Финтех | Кросс-курсы и арбитраж: USD, EUR, CNY за полгода | cbr.ru | quotes.toscrape.com | API: currencies=['USD', 'EUR', 'CNY'], days=180<br>Сайт: tag=life, pages=3 | Кросс-курсы EUR/USD и CNY/USD, корреляционная матрица, частотный анализ авторов цитат |
| 30 | E-commerce + HR (эталон преподавателя) | Оценка ниши деловой литературы: ассортимент и цены конкурента плюс доступность кадров для запуска направления | opendata.trudvsem.ru | books.toscrape.com | API: text=аналитик данных, pages=3, per_page=100<br>Сайт: category=business_35, pages=3 | Медиана и квартили цен, цены по рейтингу, доля в наличии, зарплатная вилка по регионам, топ навыков |

## Критерии оценки (до 10 баллов)

| Блок | Что оценивается | Максимум |
|---|---|---:|
| 1. Сбор данных | Корректная пагинация и остановка цикла; паузы и повторные попытки; обработка кодов ответа и исключений (`try/except`, проверка `status_code`); объём выборки не ниже минимума варианта; понятный лог сбора | 25 |
| 2. Очистка, RegEx и XPath | Рабочие селекторы CSS **и** XPath, сверка результатов; регулярные выражения с тестами; доля распознанных значений ≥ 80 %; итоговый `DataFrame` с типизированными колонками | 25 |
| 3. EDA и визуализация | Не менее четырёх графиков из варианта; подписанные оси и единицы измерения; указан объём выборки; выбраны устойчивые метрики (медиана, квартили); графики читаются без пояснений | 25 |
| 4. Бизнес-выводы | Выводы опираются на посчитанные числа; сформулированы действия, а не наблюдения; указаны ожидаемый эффект и способ проверки; честно названы ограничения | 25 |

### Шкала

| Баллы | Оценка |
|---|---|
| 8-10 | отлично |
| 7-8 | хорошо |
| 5-6 | удовлетворительно |
| < 5 | работа возвращается на доработку |

### Штрафы

| Нарушение | Штраф |
|---|---:|
| Ключ, токен или пароль в коде либо в выводе ячейки | −1,5 и обязательный отзыв ключа |
| Сбор без пауз и обработки ошибок («жёсткий» цикл по страницам) | −1,0 |
| Выводы на аварийных (синтетических) данных без указания этого факта | −1,0 |
| Разбор HTML регулярными выражениями вместо парсера | −0,5 |
| Код без комментариев, «магические числа», нарушения PEP 8 (длина строки, именование) | −0,5 |
| Графики без заголовков, подписей осей и единиц измерения | −0,5 |
| Сдан файл вместо ссылки на репозиторий; ноутбук без выводов ячеек | −1,0 |

**Дополнительно:** задание со звёздочкой в конце ноутбука — до +1,0 баллов сверх рубрики (но итог не выше 2).

In [ ]:
#@title Самопроверка перед сдачей { display-mode: "form" }
checks = []

def check(name, condition, hint=""):
    checks.append((name, bool(condition), hint))

check("Выбран вариант 1–30", 1 <= VARIANT <= 30)
check("Собраны данные через API (≥ 50 записей)", len(df_api) >= 50,
      "увеличьте число страниц в параметрах варианта")
check("Собраны данные парсингом (≥ 40 карточек)", len(df_web) >= 40,
      "пройдите больше страниц пагинации")
check("BS4 и XPath дают одинаковый результат", same == len(titles_bs4) and len(titles_bs4) > 0,
      "сравните выражения селекторов")
check("Есть числовая колонка после RegEx",
      any(pd.api.types.is_numeric_dtype(df_web[c]) for c in df_web.columns) if len(df_web) else False)
check("Ключевое числовое поле заполнено не менее чем на 80 %",
      ("price" in df_web and df_web["price"].notna().mean() >= 0.8) if len(df_web) else False)
#check("В вакансиях распознаны зарплаты", ("salary_mid" in df_api
 #     and df_api["salary_mid"].notna().sum() >= 10) if len(df_api) else False)    В 3 варианте нет вакансий
#check("Извлечены навыки из текста требований", В 3 варианте нет скиллов
 #     ("skills" in df_api and df_api["skills"].map(len).sum() > 0) if len(df_api) else False)
check("Запросы шли с паузами и ретраями", REQUEST_PAUSE >= 0.1 and len(REQUEST_LOG) > 0)
secret_like = [k for k, v in list(globals().items())
               if isinstance(v, str) and re.match(r"^(ghp_|github_pat_|sk-|AIza)[A-Za-z0-9_\-]{10,}$", v)]
check("Ключи не захардкожены в ноутбуке", not secret_like,
      "используйте get_secret() и Colab Secrets вместо переменной с ключом")
check("Данные реальные, не аварийные", API_SOURCE != "synthetic" and WEB_SOURCE != "offline-snapshot",
      "повторите сбор при доступной сети")

ok = sum(1 for _, passed, _ in checks if passed)
print(f"Выполнено: {ok} из {len(checks)}\n")
for name, passed, hint in checks:
    print(f"  {'✅' if passed else '❌'} {name}" + ("" if passed or not hint else f" — {hint}"))
if ok < len(checks):
    print("\nНезакрытые пункты снижают балл по соответствующему разделу критериев.")

Выполнено: 9 из 9

  ✅ Выбран вариант 1–30
  ✅ Собраны данные через API (≥ 50 записей)
  ✅ Собраны данные парсингом (≥ 40 карточек)
  ✅ BS4 и XPath дают одинаковый результат
  ✅ Есть числовая колонка после RegEx
  ✅ Ключевое числовое поле заполнено не менее чем на 80 %
  ✅ Запросы шли с паузами и ретраями
  ✅ Ключи не захардкожены в ноутбуке
  ✅ Данные реальные, не аварийные


## Задание со звёздочкой (до +2 баллов)

Разовый сбор данных — учебная задача. Производственная задача — **мониторинг**: данные собираются регулярно, источник меняется, а система не должна разваливаться. Ниже пять направлений; выберите одно и доведите до измеримого результата.

In [ ]:
#@title Задание со звёздочкой (до +2 балла сверх рубрики) { display-mode: "form" }
# Выберите одну задачу и решите её ниже. Каждая проверяется измеримым результатом.
#
# A. Условные запросы и инкрементальность.
#    Сохраните ETag/Last-Modified ответа, при повторном сборе отправьте заголовки
#    If-None-Match / If-Modified-Since и покажите, сколько запросов вернули 304 Not Modified
#    и сколько трафика это сэкономило.
#
# B. Параллельный сбор с соблюдением лимита.
#    Переведите обход пагинации на concurrent.futures.ThreadPoolExecutor с ограничителем
#    скорости (token bucket). Измерьте ускорение и докажите, что частота запросов
#    не превысила заданный порог.
#
# C. Дедупликация объявлений.
#    Одна вакансия часто публикуется несколько раз. Постройте дубликаты через шинглы
#    и коэффициент Жаккара, оцените долю дублей и покажите, как она искажает медиану зарплаты.
#
# D. Мониторинг вместо разового сбора.
#    Сохраните снимок в parquet с меткой времени, реализуйте второй запуск и таблицу
#    изменений (новые позиции, изменённые цены, исчезнувшие товары).
#
# E. Устойчивость парсера.
#    Напишите тесты, которые ломают парсер: отсутствующий тег, пустая цена, изменённый
#    класс. Добейтесь, чтобы парсер не падал, а помечал строку как проблемную.